# AGORA z0 adaptive DM/EM analysis

This notebook is designed to live directly under the AGORA `z0/` directory. It auto-discovers the main snapshot for `ARTI`, `Enzo`, `AREPO`, `GADGET3`, `GEAR`, and `CHANGA`, while still allowing a manual snapshot path. It calls `AGORA_parallel_DM_EM_projection_pipeline.py` in the same directory.


In [ ]:
from pathlib import Path
import sys
import importlib
import json
import csv
import numpy as np

# This notebook should be stored in the z0 directory.
ROOT = Path('/home/zhaozhang/local/AGORA_work/AGORA_Data/z0')
PIPELINE = ROOT / 'AGORA_parallel_DM_EM_projection_pipeline.py'

if not PIPELINE.exists():
    raise FileNotFoundError(f'Missing pipeline script: {PIPELINE}')

sys.path.insert(0, str(ROOT))
import AGORA_parallel_DM_EM_projection_pipeline as agora
agora = importlib.reload(agora)

print('ROOT =', ROOT)
print('pipeline =', PIPELINE)
quiver_scale = 2200
width = 0.0022
alpha = 0.75
quiver_kwargs = dict(color="black", scale=quiver_scale, width=width, alpha=alpha)

## Dataset auto-discovery

Use `discover_dataset(...)` to find the snapshot automatically from the code folder. You can pass either the folder name shown in Finder (`ARTI`, `Enzo`, `AREPO`, `GADGET3`, `GEAR`, `CHANGA`) or the normalized code name (`ART-I`, `ENZO`, `GADGET-3`, etc.).


In [ ]:
CODE_ALIASES = {
    'ARTI': 'ART-I',
    'ART-I': 'ART-I',
    'ART': 'ART-I',
    'ENZO': 'ENZO',
    'Enzo': 'ENZO',
    'AREPO': 'AREPO',
    'GADGET3': 'GADGET-3',
    'GADGET-3': 'GADGET-3',
    'GEAR': 'GEAR',
    'CHANGA': 'CHANGA',
    'G4Cal_Pablo': 'GADGET-4',
    'G4CAL_PABLO': 'GADGET-4',
    'G4Cal': 'GADGET-4',
}

FOLDER_ALIASES = {
    'ART-I': 'ARTI',
    'ENZO': 'Enzo',
    'AREPO': 'AREPO',
    'GADGET-3': 'GADGET3',
    'GADGET-4': 'G4Cal_Pablo',
    'GEAR': 'GEAR',
    'CHANGA': 'CHANGA',
    'G4Cal_Pablo': 'GADGET-4',
    'G4CAL_PABLO': 'GADGET-4',
    'G4Cal': 'GADGET-4',
}

def normalize_user_code(code):
    if code not in CODE_ALIASES:
        raise ValueError(f'Unknown code {code!r}. Use one of: {sorted(CODE_ALIASES)}')
    return CODE_ALIASES[code]

def candidate_snapshots_for_code(code, root=ROOT):
    normalized = normalize_user_code(code)
    folder = root / FOLDER_ALIASES[normalized]
    if not folder.exists():
        raise FileNotFoundError(f'Missing folder for {normalized}: {folder}')

    if normalized == 'ART-I':
        candidates = sorted(folder.glob('*.d'))
    elif normalized == 'ENZO':
        candidates = sorted(p for p in folder.glob('RD*/RD*') if p.is_file() and p.name == p.parent.name)
    elif normalized == 'AREPO':
        # Main AREPO snapshot is snap_###.hdf5. Exclude auxiliary files such as snap_###.hsml.hdf5.
        candidates = sorted(p for p in folder.glob('snap_*.hdf5') if '.hsml.' not in p.name and '.kdtree' not in p.name)
    elif normalized in {'GADGET-3', 'GADGET-4'}:
        # Multi-file Gadget snapshots should be opened from the .0.hdf5 member.
        candidates = (sorted(folder.glob('snapshot_*/*.0.hdf5')) + sorted(folder.glob('snapshot_*.hdf5')) + sorted(folder.glob('snap_*.hdf5')))
        candidates = [p for p in candidates if 'fof_subhalo' not in p.name]
    elif normalized == 'GEAR':
        candidates = sorted(p for p in (list(folder.glob('snapshot_*.hdf5')) + list(folder.glob('*.hdf5'))) if '.hsml.' not in p.name)
    elif normalized == 'CHANGA':
        # CHANGA/Tipsy main file is normally like ncal-IV.003524. Sidecars append names such as .HII or .massform.
        def is_changa_main_file(path):
            if not path.is_file() or not path.name.startswith('ncal-'):
                return False
            parts = path.name.split('.')
            return len(parts) == 2 and parts[-1].isdigit()
        preferred = sorted(p for p in folder.iterdir() if is_changa_main_file(p))
        fallback = sorted(
            p for p in folder.iterdir()
            if p.is_file() and not p.name.startswith('.') and p.name not in {'wget-log', 'robots.txt.tmp'}
            and not any(p.name.endswith(s) for s in ['.HII', '.massform', '.Metalsdot', '.ESNRate', '.kdtree'])
        )
        candidates = preferred or fallback
    else:
        candidates = []
    return normalized, folder, candidates

def discover_dataset(code, root=ROOT, index=0):
    normalized, folder, candidates = candidate_snapshots_for_code(code, root)
    if not candidates:
        raise FileNotFoundError(f'No candidate snapshot found for {normalized} in {folder}')
    snapshot = candidates[index]
    return {
        'input_code': code,
        'code': normalized,
        'folder': folder,
        'snapshot': snapshot,
        'candidate_count': len(candidates),
        'all_candidates': candidates,
    }

for name in ['ARTI', 'Enzo', 'AREPO', 'GADGET3', 'GEAR', 'CHANGA', 'G4Cal_Pablo']:
    try:
        info = discover_dataset(name)
        print(f'{name:8s} -> code={info["code"]:8s} snapshot={info["snapshot"]} candidates={info["candidate_count"]}')
    except Exception as exc:
        print(f'{name:8s} -> {exc}')


## Select one dataset and run

Change only `DATASET_CODE` for the common case. Set `MANUAL_SNAPSHOT` only if you want to override auto-discovery.


In [ ]:
DATASET_CODE = 'GEAR'       # ARTI, Enzo, AREPO, GADGET3, GEAR, CHANGA, G4Cal_Pablo
MANUAL_SNAPSHOT = None      # Example: Path('/path/to/snapshot'); keep None for auto-discovery.

N_JOBS = 14
RANDOM_OBSERVERS = 128 # Number of random observer positions in the disk plane (0 = disable)


# LOS integration settings
N_LOS = 5000 # Maximum LOS length [kpc]
DS_KPC = 0.25  #LOS integration step in kpc. Set to a small value for better accuracy, at the cost of longer runtime.
# Solar-circle radius for observer placement [kpc]
S_MAX_KPC = 250 
R_SUN_KPC = 8.2  #Only relevant if RANDOM_OBSERVERS > 0.

# Bulk velocity reference
VELOCITY_REFERENCE_SOURCE = "gas"  # Only consider this component when computing the bulk velocity. 
VELOCITY_REFERENCE_RADIUS_KPC = 30.0 #Only consider stars within this radius when computing the bulk velocity.

# ISM definition (cylindrical cut)
ISM_R_KPC = 20.0     # Radial extent [kpc]
ISM_ABS_Z_KPC = 5.0  # Vertical extent [kpc]

 # Hot gas threshold [K]
HOT_TMIN_K = 1.0e6

# Projection settings
PROJECTION_BOX_KPC = 40 #Size of the projection box in kpc. The projection will cover a square region of [-BOX, BOX] x [-BOX, BOX]. 
PROJECTION_NPIX = 512 #Number of pixels along each axis for the projection maps. The final maps will be PROJECTION_NPIX x PROJECTION_NPIX. Set to a larger value for higher resolution, at the cost of longer runtime and higher memory usage.
PROJECTION_MAX_ELEMENTS = 1000_000_000 #Maximum number of gas elements to consider when generating the projection. Set to a large value to effectively disable downsampling.
PROJECTION_QUIVER_STEP = 12 #Step size in pixels for plotting velocity quiver arrows on the projection maps. Set to a large value to effectively disable quiver plotting.
PROJECTION_LOS_HALF_THICKNESS_KPC = ISM_ABS_Z_KPC #Projection LOS half-thickness [kpc]. Set to None for full-column projection.

IONIZATION_MODE = None  # None means: G4Cal_Pablo -> temperature_weighted; other datasets -> fully_ionized.

if MANUAL_SNAPSHOT is None:
    selected = discover_dataset(DATASET_CODE)
    CODE = selected['code']
    SNAPSHOT = selected['snapshot']
else:
    CODE = normalize_user_code(DATASET_CODE)
    SNAPSHOT = Path(MANUAL_SNAPSHOT)

OUTPUT_BASE = ROOT / 'parallel_outputs'
OUTDIR = OUTPUT_BASE / FOLDER_ALIASES[CODE]
if not OUTDIR.exists():
    OUTDIR.mkdir(parents=True, exist_ok=True)

print('CODE     =', CODE)
print('SNAPSHOT =', SNAPSHOT)
ION_MODE = IONIZATION_MODE or ('temperature_weighted' if DATASET_CODE.upper().replace('-', '_') in {'G4CAL_PABLO', 'G4CAL'} else 'fully_ionized')

print('OUTDIR   =', OUTDIR)
print('ION_MODE =', ION_MODE)


## Hot X-ray-like EM diagnostics

The pipeline now also supports X-ray-style hot-gas EM diagnostics. Enable `--make-hot-em-diagnostics` to write:

- `MWlike_4pi_hot_EM_vs_GC_angle.png`: hot EM versus angular distance from the Galactic centre.
- `MWlike_4pi_hot_EM_GC_polar.png`: polar sky view centered on the Galactic-centre direction.
- `MWlike_4pi_EM_ne2_hot_pc_cm6_mollweide.png`: hot EM Mollweide map when `--make-mollweide` is enabled.

The hot-gas selection is controlled by `--hot-Tmin-K`, with default `1.0e6 K`. The CSV/NPZ output also includes `angle_from_galactic_center_deg` and `EM_ne2_hot_pc_cm6`.


In [ ]:
main_args = [
    '--snapshot', str(SNAPSHOT),
    '--code', CODE,
    '--integration-backend', 'auto',
    '--outdir', str(OUTDIR),
    '--n-los', str(N_LOS),
    '--particle-interpolation', 'auto',
    '--ionization-mode', ION_MODE,
    '--n-jobs', str(N_JOBS),
    '--chunk-los', '64',
    '--s-max-kpc', str(S_MAX_KPC),
    '--ds-kpc', str(DS_KPC),
    '--R-sun-kpc', str(R_SUN_KPC),
    '--center-mode', 'stellar_com',
    '--disk-normal-source', 'stars',
    '--velocity-reference-source', VELOCITY_REFERENCE_SOURCE,
    '--velocity-reference-radius-kpc', str(VELOCITY_REFERENCE_RADIUS_KPC),
    '--ism-R-kpc', str(ISM_R_KPC),
    '--ism-abs-z-kpc', str(ISM_ABS_Z_KPC),
    '--hot-Tmin-K', str(HOT_TMIN_K),
    '--make-mollweide',
    '--make-hot-em-diagnostics',
    '--make-projections',
    '--projection-box-kpc', str(PROJECTION_BOX_KPC),
    '--projection-npix', str(PROJECTION_NPIX),
    '--projection-max-elements', str(PROJECTION_MAX_ELEMENTS),
    '--projection-quiver-step', str(PROJECTION_QUIVER_STEP),
    '--random-observers', str(RANDOM_OBSERVERS),
    "--unit-base", "auto",
]

if PROJECTION_LOS_HALF_THICKNESS_KPC is not None:
    main_args.extend(['--projection-los-half-thickness-kpc', str(PROJECTION_LOS_HALF_THICKNESS_KPC)])

agora.main(main_args)


## Save reusable plotting data package

Run this after `agora.main(...)` finishes. It collects the LOS arrays, summary/metadata JSON, random-observer tables if present, and the saved projection-map arrays into one parameter-tagged package. The filename records the main plotting parameters: `Rf` is the face-on plotted radius (`projection_box_kpc / 2`), `RH` is the ISM half-height used by the decomposition, and `LoS` is the number of sightlines.


In [ ]:
import json
import csv
import shutil
from pathlib import Path
import numpy as np


def _format_float_for_name(x):
    x = float(x)
    if x == 0:
        return "0"
    if abs(x) >= 1e4 or abs(x) < 1e-2:
        s = f"{x:.0e}"
    elif abs(x - round(x)) < 1e-9:
        s = str(int(round(x)))
    else:
        s = f"{x:g}"
    return s.replace("+", "").replace("-", "m").replace(".", "p")


def _safe_name(x):
    return str(x).replace(" ", "").replace("/", "-").replace("_", "").replace(".", "p")


def _load_numeric_csv(path):
    path = Path(path)
    if not path.exists():
        return {}
    with path.open(newline="") as f:
        reader = csv.DictReader(f)
        rows = list(reader)
    if not rows:
        return {}
    out = {}
    for key in rows[0].keys():
        vals = []
        numeric = True
        for row in rows:
            try:
                vals.append(float(row[key]))
            except Exception:
                numeric = False
                break
        if numeric:
            out[key] = np.asarray(vals, dtype=np.float64)
    return out


def save_plotting_data_package(outdir=OUTDIR):
    outdir = Path(outdir)
    plot_data_dir = outdir / "plot_data"
    plot_data_dir.mkdir(parents=True, exist_ok=True)

    rf_kpc = PROJECTION_BOX_KPC / 2.0
    stem = (
        f"{_safe_name(CODE)}"
        f"_Rf{_format_float_for_name(rf_kpc)}kpc"
        f"_RH{_format_float_for_name(ISM_ABS_Z_KPC)}kpc"
        f"_LoS{int(N_LOS)}"
        f"_smax{_format_float_for_name(S_MAX_KPC)}kpc"
        f"_ds{_format_float_for_name(DS_KPC)}kpc"
        f"_hotT{_format_float_for_name(HOT_TMIN_K)}K"
        f"_npix{int(PROJECTION_NPIX)}"
    )

    arrays = {}
    source_files = {}

    los_npz = outdir / "MWlike_4pi_DM_EM_sightlines.npz"
    if los_npz.exists():
        with np.load(los_npz) as z:
            for key in z.files:
                arrays[f"los_{key}"] = z[key]
        source_files["los_npz"] = str(los_npz)

    projection_npz = outdir / "projection_maps_face_edge.npz"
    if projection_npz.exists():
        with np.load(projection_npz) as z:
            for key in z.files:
                arrays[f"projection_{key}"] = z[key]
        source_files["projection_npz"] = str(projection_npz)

    random_csv = outdir / "random_observer_special_los.csv"
    for key, val in _load_numeric_csv(random_csv).items():
        arrays[f"random_observer_{key}"] = val
    if random_csv.exists():
        source_files["random_observer_csv"] = str(random_csv)

    package_npz = plot_data_dir / f"{stem}_plot_data.npz"
    np.savez_compressed(package_npz, **arrays)

    # Keep human-readable LOS table too, with the same parameter-rich stem.
    los_csv = outdir / "MWlike_4pi_DM_EM_sightlines.csv"
    package_csv = None
    if los_csv.exists():
        package_csv = plot_data_dir / f"{stem}_sightlines.csv"
        shutil.copy2(los_csv, package_csv)
        source_files["los_csv"] = str(los_csv)

    metadata_paths = {
        "run_metadata": outdir / "MWlike_4pi_DM_EM_metadata.json",
        "summary": outdir / "MWlike_4pi_DM_EM_summary.json",
        "parallel_metadata": outdir / "parallel_pipeline_metadata.json",
        "projection_metadata": outdir / "projection_maps_face_edge_metadata.json",
        "random_observer_summary": outdir / "random_observer_special_los_summary.json",
    }
    embedded_json = {}
    for name, path in metadata_paths.items():
        if path.exists():
            try:
                embedded_json[name] = json.loads(path.read_text())
                source_files[name] = str(path)
            except Exception:
                embedded_json[name] = {"unparsed_path": str(path)}

    package_metadata = {
        "stem": stem,
        "code": CODE,
        "dataset_code_input": DATASET_CODE,
        "snapshot": str(SNAPSHOT),
        "outdir": str(outdir),
        "package_npz": str(package_npz),
        "package_csv": None if package_csv is None else str(package_csv),
        "parameters": {
            "Rf_kpc": rf_kpc,
            "RH_kpc": ISM_ABS_Z_KPC,
            "N_LOS": N_LOS,
            "N_JOBS": N_JOBS,
            "S_MAX_KPC": S_MAX_KPC,
            "DS_KPC": DS_KPC,
            "R_SUN_KPC": R_SUN_KPC,
            "ISM_R_KPC": ISM_R_KPC,
            "ISM_ABS_Z_KPC": ISM_ABS_Z_KPC,
            "HOT_TMIN_K": HOT_TMIN_K,
            "PROJECTION_BOX_KPC": PROJECTION_BOX_KPC,
            "PROJECTION_NPIX": PROJECTION_NPIX,
            "PROJECTION_MAX_ELEMENTS": PROJECTION_MAX_ELEMENTS,
            "PROJECTION_QUIVER_STEP": PROJECTION_QUIVER_STEP,
            "PROJECTION_LOS_HALF_THICKNESS_KPC": PROJECTION_LOS_HALF_THICKNESS_KPC,
            "ION_MODE": ION_MODE,
        },
        "npz_keys": sorted(arrays.keys()),
        "source_files": source_files,
        "embedded_json": embedded_json,
    }
    package_json = plot_data_dir / f"{stem}_plot_data_metadata.json"
    package_json.write_text(json.dumps(package_metadata, indent=2))

    print("saved plot data npz:", package_npz)
    if package_csv is not None:
        print("saved sightline csv:", package_csv)
    print("saved plot data metadata:", package_json)
    print("n arrays:", len(arrays))
    return package_npz, package_json


PLOT_DATA_NPZ, PLOT_DATA_METADATA = save_plotting_data_package()


In [ ]:
%matplotlib inline

## Reload plotting data package

Use this in a later notebook session to recreate plots without rerunning the expensive LOS integration. Set `PLOT_DATA_NPZ` manually if you want to load an older package.


In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm


def load_plotting_data_package(npz_path=PLOT_DATA_NPZ):
    npz_path = Path(npz_path)
    metadata_path = npz_path.with_name(npz_path.name.replace("_plot_data.npz", "_plot_data_metadata.json"))
    arrays = dict(np.load(npz_path))
    metadata = json.loads(metadata_path.read_text()) if metadata_path.exists() else {}
    print("loaded:", npz_path)
    print("metadata:", metadata_path)
    print("available keys:")
    for key in sorted(arrays):
        print(" ", key, arrays[key].shape)
    return arrays, metadata


# Use the bottom 'Replot From Saved Plot Data' panel for robust auto-discovery.
# plot_arrays, plot_metadata = load_plotting_data_package()

# Example 1: hot EM versus angle from Galactic centre.
if {"los_angle_from_galactic_center_deg", "los_EM_ne2_hot_pc_cm6"}.issubset(plot_arrays):
    angle = plot_arrays["los_angle_from_galactic_center_deg"]
    hot_em = 1e3 * plot_arrays["los_EM_ne2_hot_pc_cm6"]
    fig, ax = plt.subplots(figsize=(7.5, 5), constrained_layout=True)
    ax.scatter(angle, hot_em, s=12, alpha=0.65)
    ax.set_xlabel("Angle from Galactic centre [deg]")
    ax.set_ylabel(r"Hot EM [$10^{-3}$ cm$^{-6}$ pc]")
    ax.set_title(plot_metadata.get("stem", "hot EM profile"))
    ax.grid(True, alpha=0.3)
    plt.show()

# Example 2: replot saved gas surface-density projections.
if {"projection_gas_face_sigma", "projection_gas_edge_sigma", "projection_gas_face_extent", "projection_gas_edge_extent"}.issubset(plot_arrays):
    fig, axes = plt.subplots(2, 1, figsize=(7, 10), constrained_layout=True)
    for ax, key, ext_key, title in [
        (axes[0], "projection_gas_face_sigma", "projection_gas_face_extent", "Gas surface density face-on"),
        (axes[1], "projection_gas_edge_sigma", "projection_gas_edge_extent", "Gas surface density edge-on"),
    ]:
        arr = plot_arrays[key]
        extent = plot_arrays[ext_key]
        shown = np.log10(np.where(arr > 0, arr, np.nan))
        im = ax.imshow(shown, origin="lower", extent=extent, cmap="magma")
        ax.set_title(title)
        ax.set_aspect("equal")
        fig.colorbar(im, ax=ax, label="log surface density")
    plt.show()


## ISM-Only Gas and Stellar Projections

This panel rebuilds projection maps using only material inside the ISM cylinder in the face-on frame: `R_cyl <= ISM_R_KPC` and `|z| <= ISM_ABS_Z_KPC`. It is intentionally separate from the full projection maps saved by the main pipeline.


In [ ]:
import json
import importlib
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm


# Refresh the pipeline module in case this notebook kernel cached an older version.
agora = importlib.reload(agora)


ISM_PROJECTION_OUTDIR = OUTDIR / "ism_projection"
ISM_PROJECTION_OUTDIR.mkdir(parents=True, exist_ok=True)


def _finite_positive_lognorm(arr, pmin=2, pmax=98):
    arr = np.asarray(arr, dtype=float)
    pos = arr[np.isfinite(arr) & (arr > 0)]
    if pos.size == 0:
        return None
    vmin = max(np.nanpercentile(pos, pmin), 1e-30)
    vmax = max(np.nanpercentile(pos, pmax), vmin * 1.01)
    return LogNorm(vmin=vmin, vmax=vmax)


def _filter_component_to_ism(component, ism_r_kpc=ISM_R_KPC, ism_abs_z_kpc=ISM_ABS_Z_KPC):
    """Return a copy of a projection component restricted to the face-on ISM cylinder."""
    pos = component.get("pos")
    if pos is None or len(pos) == 0:
        return {k: None for k in component}
    pos = np.asarray(pos, dtype=float)
    r_cyl = np.sqrt(pos[:, 0] ** 2 + pos[:, 1] ** 2)
    mask = np.isfinite(r_cyl) & np.isfinite(pos[:, 2]) & (r_cyl <= ism_r_kpc) & (np.abs(pos[:, 2]) <= ism_abs_z_kpc)
    out = {}
    for key, val in component.items():
        if val is None:
            out[key] = None
        else:
            arr = np.asarray(val)
            out[key] = arr[mask]
    out["ism_mask_count"] = int(np.count_nonzero(mask))
    out["ism_mask_total"] = int(len(pos))
    return out


def _rect_projection_map(pos, mass=None, view="face-on", x_half_kpc=20.0, y_half_kpc=20.0, npix=512):
    """2D mass surface density map in a rectangular physical-kpc window."""
    if pos is None or len(pos) == 0:
        sigma = np.full((npix, npix), np.nan)
        return {"sigma": sigma, "extent": [-x_half_kpc, x_half_kpc, -y_half_kpc, y_half_kpc], "n_used": 0}

    pos = np.asarray(pos, dtype=float)
    if view == "face-on":
        x, y = pos[:, 0], pos[:, 1]
        xlabel, ylabel = "x_faceon [kpc]", "y_faceon [kpc]"
    elif view == "edge-on":
        x, y = pos[:, 0], pos[:, 2]
        xlabel, ylabel = "x_faceon [kpc]", "z_faceon [kpc]"
    else:
        raise ValueError("view must be face-on or edge-on")

    mask = (
        np.isfinite(x) & np.isfinite(y)
        & (x >= -x_half_kpc) & (x <= x_half_kpc)
        & (y >= -y_half_kpc) & (y <= y_half_kpc)
    )
    weights = np.ones(np.count_nonzero(mask), dtype=float) if mass is None else np.asarray(mass, dtype=float)[mask]
    mass_map, _, _ = np.histogram2d(
        x[mask], y[mask],
        bins=[npix, npix],
        range=[[-x_half_kpc, x_half_kpc], [-y_half_kpc, y_half_kpc]],
        weights=weights,
    )
    pixel_area = (2.0 * x_half_kpc / npix) * (2.0 * y_half_kpc / npix)
    return {
        "sigma": mass_map.T / pixel_area,
        "extent": [-x_half_kpc, x_half_kpc, -y_half_kpc, y_half_kpc],
        "xlabel": xlabel,
        "ylabel": ylabel,
        "n_used": int(np.count_nonzero(mask)),
    }


def make_ism_only_projection_current_dataset(
    outdir=ISM_PROJECTION_OUTDIR,
    projection_npix=PROJECTION_NPIX,
    projection_max_elements=PROJECTION_MAX_ELEMENTS,
):
    """Build gas/star projection maps using only material inside the ISM cylinder."""
    code = agora.normalize_code(CODE)
    cfg = agora.Config(
        snapshot=str(SNAPSHOT),
        code=CODE,
        outdir=str(OUTDIR),
        unit_base="auto",
        integration_backend="auto",
        center_mode="stellar_com",
        velocity_reference_source=VELOCITY_REFERENCE_SOURCE,
        velocity_reference_radius_kpc=VELOCITY_REFERENCE_RADIUS_KPC,
        disk_normal_source="stars",
        R_sun_kpc=R_SUN_KPC,
        n_los=N_LOS,
        s_max_kpc=S_MAX_KPC,
        ds_kpc=DS_KPC,
        ionization_mode=ION_MODE,
        ism_R_kpc=ISM_R_KPC,
        ism_abs_z_kpc=ISM_ABS_Z_KPC,
        hot_Tmin_K=HOT_TMIN_K,
    )

    ds = agora.load_dataset(str(SNAPSHOT), code, cfg.unit_base)
    code_cfg = agora.AGORA_CODE_CONFIG.get(code, agora.default_config_for_unknown(code, ds))
    agora.add_common_derived_fields(ds, code, code_cfg)

    arrays = agora.load_geometry_arrays(cfg, ds, code_cfg)
    cfg = agora.apply_dataset_specific_geometry_fallbacks(cfg, code, arrays)
    center = agora.determine_center(
        cfg,
        arrays["gas_pos"], arrays["gas_mass"], arrays["gas_rho"],
        arrays["star_pos"], arrays["star_mass"],
    )
    disk_normal = agora.compute_angular_momentum_normal(
        cfg,
        arrays["gas_pos"], arrays["gas_vel"], arrays["gas_mass"], arrays["gas_temp"],
        arrays["star_pos"], arrays["star_vel"], arrays["star_mass"],
        center,
    )
    bulk_velocity = agora.compute_bulk_velocity(
        cfg,
        arrays["gas_pos"], arrays["gas_vel"], arrays["gas_mass"],
        arrays["star_pos"], arrays["star_vel"], arrays["star_mass"],
        center,
    )
    R_faceon = agora.rotation_matrix_from_vectors(disk_normal, np.array([0.0, 0.0, 1.0]))

    gas, stars = agora.load_projection_arrays(
        cfg,
        ds,
        code_cfg,
        arrays,
        center,
        R_faceon,
        projection_max_elements,
        bulk_velocity_original=bulk_velocity,
    )
    gas_ism = _filter_component_to_ism(gas)
    stars_ism = _filter_component_to_ism(stars)

    gas_face = _rect_projection_map(
        gas_ism["pos"], gas_ism.get("mass"),
        view="face-on", x_half_kpc=ISM_R_KPC, y_half_kpc=ISM_R_KPC,
        npix=projection_npix,
    )
    gas_edge = _rect_projection_map(
        gas_ism["pos"], gas_ism.get("mass"),
        view="edge-on", x_half_kpc=ISM_R_KPC, y_half_kpc=ISM_ABS_Z_KPC,
        npix=projection_npix,
    )
    star_face = _rect_projection_map(
        stars_ism["pos"], stars_ism.get("mass"),
        view="face-on", x_half_kpc=ISM_R_KPC, y_half_kpc=ISM_R_KPC,
        npix=projection_npix,
    )
    star_edge = _rect_projection_map(
        stars_ism["pos"], stars_ism.get("mass"),
        view="edge-on", x_half_kpc=ISM_R_KPC, y_half_kpc=ISM_ABS_Z_KPC,
        npix=projection_npix,
    )

    panels = [
        (gas_face, "Gas ISM surface density (face-on)", "magma"),
        (gas_edge, "Gas ISM surface density (edge-on)", "magma"),
        (star_face, "Stellar ISM surface density (face-on)", "inferno"),
        (star_edge, "Stellar ISM surface density (edge-on)", "inferno"),
    ]
    fig, axes = plt.subplots(2, 2, figsize=(12.5, 10.5), constrained_layout=True)
    for ax, (maps, title, cmap) in zip(axes.ravel(), panels):
        sigma = np.asarray(maps["sigma"], dtype=float)
        im = ax.imshow(
            sigma,
            origin="lower",
            extent=maps["extent"],
            cmap=cmap,
            norm=_finite_positive_lognorm(sigma),
            interpolation="nearest",
        )
        ax.set_title(f"{title}, N={maps['n_used']}")
        ax.set_xlabel(maps.get("xlabel", "kpc"))
        ax.set_ylabel(maps.get("ylabel", "kpc"))
        ax.set_aspect("auto" if "edge-on" in title else "equal")
        fig.colorbar(im, ax=ax, label=r"surface density [$M_\odot$ kpc$^{-2}$]")

    out_png = Path(outdir) / "ISM_only_gas_star_projection_face_edge.png"
    fig.savefig(out_png, dpi=220, bbox_inches="tight")
    plt.show()

    out_npz = Path(outdir) / "ISM_only_gas_star_projection_face_edge.npz"
    np.savez_compressed(
        out_npz,
        gas_face_sigma=gas_face["sigma"],
        gas_edge_sigma=gas_edge["sigma"],
        star_face_sigma=star_face["sigma"],
        star_edge_sigma=star_edge["sigma"],
        gas_face_extent=np.asarray(gas_face["extent"], dtype=float),
        gas_edge_extent=np.asarray(gas_edge["extent"], dtype=float),
        star_face_extent=np.asarray(star_face["extent"], dtype=float),
        star_edge_extent=np.asarray(star_edge["extent"], dtype=float),
        gas_ism_count=np.asarray(gas_ism.get("ism_mask_count", 0), dtype=np.int64),
        gas_total_count=np.asarray(gas_ism.get("ism_mask_total", 0), dtype=np.int64),
        star_ism_count=np.asarray(stars_ism.get("ism_mask_count", 0), dtype=np.int64),
        star_total_count=np.asarray(stars_ism.get("ism_mask_total", 0), dtype=np.int64),
    )
    meta = {
        "code": CODE,
        "snapshot": str(SNAPSHOT),
        "out_png": str(out_png),
        "out_npz": str(out_npz),
        "ISM_R_KPC": float(ISM_R_KPC),
        "ISM_ABS_Z_KPC": float(ISM_ABS_Z_KPC),
        "projection_npix": int(projection_npix),
        "projection_max_elements": int(projection_max_elements),
        "center_kpc_original_frame": np.asarray(center, dtype=float).tolist(),
        "disk_normal_original_frame": np.asarray(disk_normal, dtype=float).tolist(),
        "dataset_units": agora.dataset_unit_metadata(ds),
        "gas_ism_count": int(gas_ism.get("ism_mask_count", 0)),
        "gas_total_count": int(gas_ism.get("ism_mask_total", 0)),
        "star_ism_count": int(stars_ism.get("ism_mask_count", 0)),
        "star_total_count": int(stars_ism.get("ism_mask_total", 0)),
        "surface_density_units": "Msun / physical kpc^2",
        "ism_cut": "sqrt(x_faceon^2 + y_faceon^2) <= ISM_R_KPC and abs(z_faceon) <= ISM_ABS_Z_KPC",
    }
    out_json = Path(outdir) / "ISM_only_gas_star_projection_face_edge_metadata.json"
    out_json.write_text(json.dumps(meta, indent=2))

    print("Saved:", out_png)
    print("Saved:", out_npz)
    print("Saved:", out_json)
    print(f"Gas ISM elements: {meta['gas_ism_count']} / {meta['gas_total_count']}")
    print(f"Star ISM elements: {meta['star_ism_count']} / {meta['star_total_count']}")
    return meta


# Run this cell after DATASET_CODE/SNAPSHOT/OUTDIR are set.
ism_projection_metadata = make_ism_only_projection_current_dataset()


## Batch run all discovered datasets

This cell is commented by default. Uncomment only on a server or when you really want to run every code.


In [ ]:
# ALL_CODES = ['ARTI', 'Enzo', 'AREPO', 'GADGET3', 'GEAR', 'CHANGA', 'G4Cal_Pablo']
# for dataset_code in ALL_CODES:
#     selected = discover_dataset(dataset_code)
#     code = selected['code']
#     snapshot = selected['snapshot']
#     outdir = ROOT / 'parallel_outputs' / FOLDER_ALIASES[code]
#     print('Running', dataset_code, snapshot)
#     agora.main([
#         '--snapshot', str(snapshot),
#         '--code', code,
#         '--integration-backend', 'auto',
#         '--outdir', str(outdir),
#         '--n-los', str(N_LOS),
#         '--particle-interpolation', 'auto',
#         '--ionization-mode', 'temperature_weighted' if dataset_code in {'G4Cal_Pablo', 'G4Cal'} else 'fully_ionized',
#         '--n-jobs', str(N_JOBS),
#         '--chunk-los', '64',
#         '--s-max-kpc', '250',
#         '--ds-kpc', '0.25',
#         '--R-sun-kpc', '8.2',
#         '--make-mollweide',
#         '--make-hot-em-diagnostics',
#         '--make-projections',
#         '--projection-box-kpc', str(PROJECTION_BOX_KPC),
#         '--projection-npix', str(PROJECTION_NPIX),
#         '--projection-max-elements', str(PROJECTION_MAX_ELEMENTS),
#         '--random-observers', str(RANDOM_OBSERVERS),
#     ])


## Locate outputs

After a run, this cell lists the files produced for the selected code.


In [ ]:
if OUTDIR.exists():
    for p in sorted(OUTDIR.iterdir()):
        print(p.name)
else:
    print('Output directory does not exist yet:', OUTDIR)


## Shared Results Plotting Utilities

Run this once before the comparison, projection, Mollweide, hot-EM, and PDF plotting panels.


In [ ]:
from pathlib import Path
import json
import csv
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

# Shared plotting helpers used by all result-viewer panels below.
ROOT = Path(globals().get('ROOT', '/home/zhaozhang/local/AGORA_work/AGORA_Data/z0'))
OUTROOT = ROOT / 'parallel_outputs'
try:
    SELECTED_CODE = Path(OUTDIR).name
except Exception:
    SELECTED_CODE = globals().get('SELECTED_CODE', 'G4Cal_Pablo')
SELECTED_OUTDIR = OUTROOT / SELECTED_CODE
FIGURE_DIR = OUTROOT / 'viewer_figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


# By default, keep only production output directories in comparison plots.
# Smoke/debug/fix directories are useful for testing but should not enter science figures.
EXCLUDE_OUTPUT_NAME_PARTS = tuple(globals().get('EXCLUDE_OUTPUT_NAME_PARTS', ('_smoke', '_debug', '_fix')))

def is_production_output_name(name):
    return not any(part in str(name) for part in EXCLUDE_OUTPUT_NAME_PARTS)

def is_production_output_dir(path):
    path = Path(path)
    return path.is_dir() and is_production_output_name(path.name)

def load_json(path):
    path = Path(path)
    return json.loads(path.read_text()) if path.exists() else None

def safe_filename(name):
    return ''.join(c if c.isalnum() or c in '._-' else '_' for c in str(name)).strip('_')

def save_figure(fig, name, dpi=220):
    path = FIGURE_DIR / f'{safe_filename(name)}.png'
    fig.savefig(path, dpi=dpi, bbox_inches='tight')
    print('Saved figure:', path)
    return path

def load_json(path):
    path = Path(path)
    return json.loads(path.read_text()) if path.exists() else None

def _find_nested_value(obj, names):
    if isinstance(obj, dict):
        for key, val in obj.items():
            if key in names:
                return val
            found = _find_nested_value(val, names)
            if found is not None:
                return found
    elif isinstance(obj, (list, tuple)):
        for val in obj:
            found = _find_nested_value(val, names)
            if found is not None:
                return found
    return None

def code_redshift(code, outroot=OUTROOT):
    code = str(code)
    for fname in ['parallel_pipeline_metadata.json', 'MWlike_4pi_DM_EM_metadata.json', 'projection_maps_face_edge_metadata.json']:
        meta = load_json(Path(outroot) / code / fname)
        if not meta:
            continue
        z = _find_nested_value(meta, {'current_redshift', 'redshift', 'Redshift', 'z'})
        if z is not None:
            try:
                z = float(z)
                if np.isfinite(z):
                    return z
            except Exception:
                pass
    # Some old output metadata was written before dataset_units were recorded.
    fallback = {'GADGET3': 0.3, 'GEAR': 0.2971321854409694}
    return fallback.get(code)

def code_label_with_redshift(code, z_threshold=0.05):
    z = code_redshift(code)
    if z is not None and abs(z) >= z_threshold:
        return f'{code}\n(z={z:.2f})'
    return str(code)


print('OUTROOT =', OUTROOT)
print('SELECTED_CODE =', SELECTED_CODE)
print('FIGURE_DIR =', FIGURE_DIR)


## Compare outputs across simulation codes

Run this after one or more per-code output directories exist under `parallel_outputs/`. It reads each `MWlike_4pi_DM_EM_summary.json` and compares the median plus 16-84 percentile range for DM, total EM, and hot-gas EM (`T >= --hot-Tmin-K`).


In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

OUTROOT = ROOT / 'parallel_outputs'
summary_rows = []
for d in sorted(OUTROOT.glob('*')):
    if not is_production_output_dir(d):
        continue
    summary_file = d / 'MWlike_4pi_DM_EM_summary.json'
    random_file = d / 'random_observer_special_los_summary.json'
    if not summary_file.exists():
        continue
    summary = json.loads(summary_file.read_text())
    row = {
        'code': d.name,
        'DM_median': summary['DM_total_pc_cm3']['median'],
        'DM_p16': summary['DM_total_pc_cm3']['p16'],
        'DM_p84': summary['DM_total_pc_cm3']['p84'],
        'EM_median': summary['EM_ne2_total_pc_cm6']['median'],
        'EM_p16': summary['EM_ne2_total_pc_cm6']['p16'],
        'EM_p84': summary['EM_ne2_total_pc_cm6']['p84'],
        'has_random_observer_summary': random_file.exists(),
    }
    if 'EM_ne2_hot_pc_cm6' in summary:
        row.update({
            'HotEM_median': summary['EM_ne2_hot_pc_cm6']['median'],
            'HotEM_p16': summary['EM_ne2_hot_pc_cm6']['p16'],
            'HotEM_p84': summary['EM_ne2_hot_pc_cm6']['p84'],
        })
    summary_rows.append(row)

summary_rows


In [ ]:
if summary_rows:
    labels = [r['code'] for r in summary_rows]
    x = np.arange(len(labels))

    panels = [
        ('DM_total', 'DM_median', 'DM_p16', 'DM_p84', 'DM [pc cm^-3]', 'tab:blue'),
        ('EM_total', 'EM_median', 'EM_p16', 'EM_p84', 'EM [pc cm^-6]', 'tab:red'),
    ]
    if all('HotEM_median' in r for r in summary_rows):
        panels.append(('EM_hot', 'HotEM_median', 'HotEM_p16', 'HotEM_p84', 'Hot EM [pc cm^-6]', 'tab:orange'))

    fig, axes = plt.subplots(len(panels), 1, figsize=(8, 3.2 * len(panels)), constrained_layout=True)
    axes = np.atleast_1d(axes)
    for ax, (_, med_key, p16_key, p84_key, ylabel, color) in zip(axes, panels):
        med = np.array([r[med_key] for r in summary_rows], dtype=float)
        lo = med - np.array([r[p16_key] for r in summary_rows], dtype=float)
        hi = np.array([r[p84_key] for r in summary_rows], dtype=float) - med
        ax.errorbar(x, med, yerr=[lo, hi], fmt='o', capsize=4, color=color)
        ax.set_yscale('log')
        ax.set_ylabel(ylabel)
        ax.set_xticks(x, labels, rotation=30, ha='right')
        ax.grid(True, alpha=0.3)

    OUTROOT.mkdir(exist_ok=True)
    fig.savefig(OUTROOT / 'comparison_DM_EM_hotEM_median_p16_p84.png', dpi=180)
    plt.show()
else:
    print('No per-code summary files found under', OUTROOT)


## PDF Distribution Fits and Goodness-of-Fit

This panel reads the saved per-sightline DM/EM arrays for each code, plots their PDF distributions, and fits Log-normal, 3-component Gaussian mixture in log10 space, Weibull, and a high-tail GPD model. The GPD fit is applied only to the upper tail above the 90th percentile, so its goodness metrics describe the tail rather than the full distribution.


In [ ]:
import csv
import json
import math
import re
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats, special

# Plot style requested for publication-like panels.
font = {"family": "serif", "size": 16}
font_label = {"family": "serif", "size": 20}
tick_size = 18
font_legend = {"family": "serif", "size": 16}
font_legend_small = {"family": "serif", "size": 12}

plt.rc("font", **font)
plt.rcParams.update({
    "axes.labelsize": font_label["size"],
    "axes.titlesize": font_label["size"],
    "xtick.labelsize": tick_size,
    "ytick.labelsize": tick_size,
    "legend.fontsize": font_legend_small["size"],
})

OUTROOT = ROOT / "parallel_outputs"
FIT_OUTPUT_DIR = OUTROOT / "pdf_fit_figures"
FIT_OUTPUT_CSV = OUTROOT / "distribution_fit_goodness.csv"

FIT_QUANTITIES = {
    "DM_total_pc_cm3": r"DM [pc cm$^{-3}$]",
    "EM_ne2_total_pc_cm6": r"EM [pc cm$^{-6}$]",
    "EM_ne2_hot_pc_cm6": r"Hot EM [pc cm$^{-6}$]",
}

MODEL_STYLES = {
    "Log-normal": {"color": "#1f77b4", "linestyle": "-"},
    "GMM": {"color": "#d62728", "linestyle": "--"},
    "Weibull": {"color": "#2ca02c", "linestyle": ":"},
    "GPD": {"color": "#9467bd", "linestyle": "-."},
}


def safe_name(name):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(name)).strip("_")


def load_per_code_distributions(outroot=OUTROOT):
    """Load positive finite per-sightline arrays from each code output directory."""
    per_code = {}
    for npz_path in sorted(Path(outroot).glob("*/MWlike_4pi_DM_EM_sightlines.npz")):
        code_name = npz_path.parent.name
        with np.load(npz_path) as data:
            arrays = {}
            for key in FIT_QUANTITIES:
                if key not in data.files:
                    continue
                vals = np.asarray(data[key], dtype=float)
                vals = vals[np.isfinite(vals) & (vals > 0)]
                if vals.size > 5:
                    arrays[key] = vals
        if arrays:
            per_code[code_name] = arrays
    return per_code


def fitted_ad_statistic(x, cdf_func):
    """Anderson-Darling style statistic for any fitted CDF."""
    x = np.sort(np.asarray(x, dtype=float))
    n = len(x)
    if n < 2:
        return np.nan
    F = np.clip(cdf_func(x), 1e-12, 1.0 - 1e-12)
    i = np.arange(1, n + 1, dtype=float)
    return float(-n - np.mean((2.0 * i - 1.0) * (np.log(F) + np.log(1.0 - F[::-1]))))


def summarize_fit_metrics(x, model_name, n_params, loglike, cdf_func, fit_domain="full"):
    """Return common goodness-of-fit metrics for a fitted model."""
    x = np.asarray(x, dtype=float)
    n = len(x)
    if n <= n_params + 1:
        aic = bic = np.nan
    else:
        aic = 2.0 * n_params - 2.0 * loglike
        bic = n_params * np.log(n) - 2.0 * loglike
    try:
        ks_stat, ks_p = stats.kstest(x, cdf_func)
    except Exception:
        ks_stat, ks_p = np.nan, np.nan
    try:
        ad = fitted_ad_statistic(x, cdf_func)
    except Exception:
        ad = np.nan
    return {
        "model": model_name,
        "fit_domain": fit_domain,
        "n_fit": int(n),
        "n_params": int(n_params),
        "loglike": float(loglike),
        "AIC": float(aic),
        "BIC": float(bic),
        "KS_stat": float(ks_stat),
        "KS_p": float(ks_p),
        "AD_stat": float(ad),
    }


def fit_lognormal(x):
    shape, loc, scale = stats.lognorm.fit(x, floc=0.0)
    dist = stats.lognorm(shape, loc=loc, scale=scale)
    loglike = float(np.sum(dist.logpdf(x)))
    fit = summarize_fit_metrics(x, "Log-normal", 2, loglike, dist.cdf)
    fit["params"] = json.dumps({"sigma_ln": shape, "loc": loc, "scale": scale})
    return fit, dist.pdf


def fit_weibull(x):
    shape, loc, scale = stats.weibull_min.fit(x, floc=0.0)
    dist = stats.weibull_min(shape, loc=loc, scale=scale)
    loglike = float(np.sum(dist.logpdf(x)))
    fit = summarize_fit_metrics(x, "Weibull", 2, loglike, dist.cdf)
    fit["params"] = json.dumps({"shape": shape, "loc": loc, "scale": scale})
    return fit, dist.pdf


def fit_gpd_tail(x, percentile=90.0):
    """Fit a GPD to excesses above a high threshold and scale the PDF by tail fraction."""
    threshold = float(np.nanpercentile(x, percentile))
    tail = x[x >= threshold]
    if len(tail) < 8 or not np.isfinite(threshold):
        raise ValueError("Too few tail samples for GPD")
    excess = tail - threshold
    shape, loc, scale = stats.genpareto.fit(excess, floc=0.0)
    dist = stats.genpareto(shape, loc=loc, scale=scale)
    tail_fraction = len(tail) / len(x)
    loglike = float(np.sum(dist.logpdf(excess)))
    fit = summarize_fit_metrics(
        excess,
        "GPD tail",
        2,
        loglike,
        dist.cdf,
        fit_domain=f"tail >= p{percentile:.0f}",
    )
    fit["params"] = json.dumps({
        "threshold": threshold,
        "shape_k": shape,
        "loc": loc,
        "scale_sigma": scale,
        "tail_fraction": tail_fraction,
    })

    def pdf(xx):
        xx = np.asarray(xx, dtype=float)
        yy = np.full_like(xx, np.nan, dtype=float)
        m = xx >= threshold
        yy[m] = tail_fraction * dist.pdf(xx[m] - threshold)
        return yy

    return fit, pdf


def fit_gmm_log10_1d(x, n_components=3, max_iter=400, tol=1e-7, seed=12345):
    """Fit a 1D Gaussian mixture to log10(x) using EM, then transform PDF/CDF back to x."""
    y = np.log10(np.asarray(x, dtype=float))
    y = y[np.isfinite(y)]
    n = len(y)
    k = min(int(n_components), max(1, n // 10))
    if k < 1:
        raise ValueError("Too few samples for GMM")

    weights = np.full(k, 1.0 / k)
    means = np.quantile(y, np.linspace(0.15, 0.85, k))
    base_var = np.nanvar(y) if np.nanvar(y) > 0 else 1.0
    variances = np.full(k, max(base_var / k, 1e-4))

    prev_ll = -np.inf
    for _ in range(max_iter):
        log_resp = np.column_stack([
            np.log(weights[j] + 1e-300) + stats.norm.logpdf(y, means[j], math.sqrt(variances[j]))
            for j in range(k)
        ])
        log_norm = special.logsumexp(log_resp, axis=1)
        ll = float(np.sum(log_norm))
        resp = np.exp(log_resp - log_norm[:, None])
        Nk = np.sum(resp, axis=0) + 1e-300
        weights = Nk / n
        means = np.sum(resp * y[:, None], axis=0) / Nk
        variances = np.sum(resp * (y[:, None] - means[None, :]) ** 2, axis=0) / Nk
        variances = np.maximum(variances, 1e-6)
        if abs(ll - prev_ll) < tol * (1.0 + abs(prev_ll)):
            break
        prev_ll = ll

    sigmas = np.sqrt(variances)

    def log_pdf_x(xx):
        xx = np.asarray(xx, dtype=float)
        yy = np.log10(xx)
        log_py = special.logsumexp(np.column_stack([
            np.log(weights[j] + 1e-300) + stats.norm.logpdf(yy, means[j], sigmas[j])
            for j in range(k)
        ]), axis=1)
        return log_py - np.log(xx * np.log(10.0))

    def pdf_x(xx):
        return np.exp(log_pdf_x(xx))

    def cdf_x(xx):
        yy = np.log10(np.asarray(xx, dtype=float))
        out = np.zeros_like(yy, dtype=float)
        for w, mu, sig in zip(weights, means, sigmas):
            out += w * stats.norm.cdf(yy, mu, sig)
        return out

    loglike_x = float(np.sum(log_pdf_x(x)))
    n_params = (k - 1) + k + k
    fit = summarize_fit_metrics(x, f"GMM-{k} log10", n_params, loglike_x, cdf_x)
    order = np.argsort(means)
    fit["params"] = json.dumps({
        "weights": weights[order].tolist(),
        "mu_log10": means[order].tolist(),
        "sigma_log10": sigmas[order].tolist(),
    })
    return fit, pdf_x


def fit_all_models(x):
    """Fit all requested models and return metric rows plus PDF callables."""
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x) & (x > 0)]
    models = []
    fitters = [fit_lognormal, fit_gmm_log10_1d, fit_weibull, fit_gpd_tail]
    for fitter in fitters:
        try:
            row, pdf = fitter(x)
            models.append((row, pdf))
        except Exception as exc:
            models.append(({"model": fitter.__name__, "error": repr(exc)}, None))
    return models


def quantity_bins(per_code, quantity, n_bins=56, pmin=0.2, pmax=99.8):
    vals = [arrays[quantity] for arrays in per_code.values() if quantity in arrays]
    if not vals:
        return None, None
    all_vals = np.concatenate(vals)
    all_vals = all_vals[np.isfinite(all_vals) & (all_vals > 0)]
    xmin = max(np.nanpercentile(all_vals, pmin), np.nanmin(all_vals))
    xmax = np.nanpercentile(all_vals, pmax)
    if not np.isfinite(xmin) or not np.isfinite(xmax) or xmin <= 0 or xmax <= xmin:
        xmin, xmax = np.nanmin(all_vals), np.nanmax(all_vals)
    bins = np.logspace(np.log10(xmin), np.log10(xmax), n_bins)
    xgrid = np.logspace(np.log10(xmin), np.log10(xmax), 700)
    return bins, xgrid


def available_fit_quantities(per_code):
    return [key for key in FIT_QUANTITIES if any(key in arrays for arrays in per_code.values())]


def plot_histograms_per_code(per_code, outdir=FIT_OUTPUT_DIR):
    """Save one histogram-only figure for each simulation code."""
    outdir.mkdir(parents=True, exist_ok=True)
    quantities = available_fit_quantities(per_code)
    saved = []
    for code_name in sorted(per_code):
        code_quantities = [q for q in quantities if q in per_code[code_name]]
        if not code_quantities:
            continue
        fig, axes = plt.subplots(
            len(code_quantities), 1,
            figsize=(9.5, 4.2 * len(code_quantities)),
            constrained_layout=True,
        )
        axes = np.atleast_1d(axes)
        for ax, quantity in zip(axes, code_quantities):
            bins, _ = quantity_bins({code_name: per_code[code_name]}, quantity)
            vals = per_code[code_name][quantity]
            ax.hist(vals, bins=bins, density=True, histtype="stepfilled", alpha=0.32, color="#4c78a8")
            ax.hist(vals, bins=bins, density=True, histtype="step", lw=1.8, color="#1f4e79", label=f"data (N={len(vals)})")
            ax.set_xscale("log")
            ax.set_yscale("log")
            ax.set_xlabel(FIT_QUANTITIES[quantity], fontdict=font_label)
            ax.set_ylabel("PDF", fontdict=font_label)
            ax.set_title(f"{code_name}: histogram only - {quantity}", fontdict=font_label)
            ax.grid(True, which="both", alpha=0.25)
            ax.legend(frameon=False, prop=font_legend_small)
        outfile = outdir / f"pdf_hist_only_{safe_name(code_name)}.png"
        fig.savefig(outfile, dpi=220, bbox_inches="tight")
        plt.show()
        saved.append(outfile)
        print("Saved:", outfile)
    return saved


def plot_all_code_histogram_comparison(per_code, outdir=FIT_OUTPUT_DIR):
    """Save one combined figure comparing data PDFs only, without any fit curves."""
    outdir.mkdir(parents=True, exist_ok=True)
    quantities = available_fit_quantities(per_code)
    if not quantities:
        raise RuntimeError("No saved DM/EM sightline arrays were found for PDF plotting.")
    colors = plt.cm.tab10(np.linspace(0, 1, max(3, len(per_code))))
    code_colors = {code: colors[i] for i, code in enumerate(sorted(per_code))}
    fig, axes = plt.subplots(
        len(quantities), 1,
        figsize=(10.5, 4.6 * len(quantities)),
        constrained_layout=True,
    )
    axes = np.atleast_1d(axes)
    for ax, quantity in zip(axes, quantities):
        bins, _ = quantity_bins(per_code, quantity)
        for code_name in sorted(per_code):
            if quantity not in per_code[code_name]:
                continue
            vals = per_code[code_name][quantity]
            ax.hist(
                vals,
                bins=bins,
                density=True,
                histtype="step",
                lw=2.0,
                color=code_colors[code_name],
                label=f"{code_name} (N={len(vals)})",
            )
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlabel(FIT_QUANTITIES[quantity], fontdict=font_label)
        ax.set_ylabel("PDF", fontdict=font_label)
        ax.set_title(f"Data PDF comparison: {quantity}", fontdict=font_label)
        ax.grid(True, which="both", alpha=0.25)
        ax.legend(ncol=2, frameon=False, prop=font_legend_small)
    outfile = outdir / "pdf_hist_all_codes_comparison.png"
    fig.savefig(outfile, dpi=220, bbox_inches="tight")
    plt.show()
    print("Saved:", outfile)
    return outfile


def plot_fits_per_code(per_code, outdir=FIT_OUTPUT_DIR, output_csv=FIT_OUTPUT_CSV):
    """Save one figure per code comparing fitted distributions for that code only."""
    outdir.mkdir(parents=True, exist_ok=True)
    quantities = available_fit_quantities(per_code)
    fit_rows = []
    saved = []
    for code_name in sorted(per_code):
        code_quantities = [q for q in quantities if q in per_code[code_name]]
        if not code_quantities:
            continue
        fig, axes = plt.subplots(
            len(code_quantities), 1,
            figsize=(10.5, 4.8 * len(code_quantities)),
            constrained_layout=True,
        )
        axes = np.atleast_1d(axes)
        for ax, quantity in zip(axes, code_quantities):
            vals = per_code[code_name][quantity]
            bins, xgrid = quantity_bins({code_name: per_code[code_name]}, quantity)
            ax.hist(vals, bins=bins, density=True, histtype="stepfilled", alpha=0.22, color="0.65", label=f"data (N={len(vals)})")
            ax.hist(vals, bins=bins, density=True, histtype="step", lw=1.4, color="0.25")
            for row, pdf in fit_all_models(vals):
                row.update({"code": code_name, "quantity": quantity})
                fit_rows.append(row)
                if pdf is None:
                    continue
                model = row["model"]
                style_key = "GMM" if model.startswith("GMM") else model.split()[0]
                style = MODEL_STYLES.get(style_key, {"color": "black", "linestyle": "-"})
                yfit = pdf(xgrid)
                valid = np.isfinite(yfit) & (yfit > 0)
                if np.any(valid):
                    ax.plot(
                        xgrid[valid],
                        yfit[valid],
                        lw=2.2,
                        alpha=0.95,
                        label=model,
                        **style,
                    )
            ax.set_xscale("log")
            ax.set_yscale("log")
            ax.set_xlabel(FIT_QUANTITIES[quantity], fontdict=font_label)
            ax.set_ylabel("PDF", fontdict=font_label)
            ax.set_title(f"{code_name}: data and fitted distributions - {quantity}", fontdict=font_label)
            ax.grid(True, which="both", alpha=0.25)
            ax.legend(ncol=2, frameon=False, prop=font_legend_small)
        outfile = outdir / f"pdf_fits_{safe_name(code_name)}.png"
        fig.savefig(outfile, dpi=220, bbox_inches="tight")
        plt.show()
        saved.append(outfile)
        print("Saved:", outfile)

    fieldnames = [
        "code", "quantity", "model", "fit_domain", "n_fit", "n_params",
        "loglike", "AIC", "BIC", "KS_stat", "KS_p", "AD_stat", "params", "error",
    ]
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    with open(output_csv, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        for row in fit_rows:
            writer.writerow(row)
    print("Saved goodness-of-fit table:", output_csv)
    return fit_rows, saved


def plot_pdf_analysis(per_code, outdir=FIT_OUTPUT_DIR, output_csv=FIT_OUTPUT_CSV):
    """Run the separated PDF plotting workflow requested for readability."""
    if not per_code:
        raise RuntimeError("No saved DM/EM sightline arrays were found for PDF fitting.")
    hist_files = plot_histograms_per_code(per_code, outdir)
    comparison_file = plot_all_code_histogram_comparison(per_code, outdir)
    fit_rows, fit_files = plot_fits_per_code(per_code, outdir, output_csv)
    return {
        "hist_files": hist_files,
        "comparison_file": comparison_file,
        "fit_files": fit_files,
        "fit_rows": fit_rows,
    }


per_code_distributions = load_per_code_distributions(OUTROOT)
print("Loaded codes:", sorted(per_code_distributions))
pdf_plot_outputs = plot_pdf_analysis(per_code_distributions)
fit_goodness_rows = pdf_plot_outputs["fit_rows"]

# Compact preview of the saved goodness-of-fit table.
try:
    import pandas as pd
    fit_goodness = pd.DataFrame(fit_goodness_rows)
    display_cols = ["code", "quantity", "model", "fit_domain", "n_fit", "AIC", "BIC", "KS_stat", "KS_p", "AD_stat"]
    display(fit_goodness[display_cols].sort_values(["quantity", "code", "AIC"]).head(30))
except Exception:
    for row in fit_goodness_rows[:12]:
        print({k: row.get(k) for k in ["code", "quantity", "model", "AIC", "BIC", "KS_stat", "KS_p", "AD_stat"]})


## Replot From Saved Plot Data

Use this panel when you want to redraw figures without rerunning `agora.main(...)`. It loads a parameter-tagged `*_plot_data.npz` package from `OUTDIR / plot_data/`, or from a manual path, and writes regenerated figures into `replot_from_saved/`.


In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, TwoSlopeNorm

# Set this to a specific *_plot_data.npz file if you do not want auto-discovery.
MANUAL_PLOT_DATA_NPZ = None


def find_latest_plot_data(outdir=OUTDIR):
    outdir = Path(outdir)
    search_dirs = [outdir / "plot_data"]
    try:
        search_dirs.append(LEGACY_OUTPUT_BASE / FOLDER_ALIASES[CODE] / "plot_data")
    except NameError:
        pass
    search_dirs.extend([
        ROOT / "debug_outputs" / FOLDER_ALIASES.get(CODE, str(CODE)) / "plot_data",
        ROOT / "ARTI" / "MW_DM_EM_ARTI_output" / "plot_data",
    ])
    seen = set()
    candidates = []
    for plot_dir in search_dirs:
        plot_dir = Path(plot_dir)
        if plot_dir in seen:
            continue
        seen.add(plot_dir)
        candidates.extend(plot_dir.glob("*_plot_data.npz"))
    candidates = sorted(candidates, key=lambda p: p.stat().st_mtime)
    if not candidates:
        checked = "\n".join(str(d) for d in search_dirs)
        raise FileNotFoundError(f"No *_plot_data.npz files found. Checked:\n{checked}\nRun the save plotting data cell first.")
    print("Using plot-data package:", candidates[-1])
    return candidates[-1]


def load_saved_plot_data(npz_path=None):
    npz_path = Path(npz_path) if npz_path is not None else find_latest_plot_data()
    metadata_path = npz_path.with_name(npz_path.name.replace("_plot_data.npz", "_plot_data_metadata.json"))
    arrays = dict(np.load(npz_path))
    metadata = json.loads(metadata_path.read_text()) if metadata_path.exists() else {}
    replot_dir = npz_path.parent / "replot_from_saved"
    replot_dir.mkdir(parents=True, exist_ok=True)
    print("Loaded plot data:", npz_path)
    print("Loaded metadata :", metadata_path if metadata_path.exists() else "<missing>")
    print("Replot output   :", replot_dir)
    print("Keys:")
    for key in sorted(arrays):
        print(" ", key, arrays[key].shape)
    return arrays, metadata, replot_dir


def positive_lognorm(vals, pmin=2, pmax=98):
    vals = np.asarray(vals, dtype=float)
    positive = vals[np.isfinite(vals) & (vals > 0)]
    if positive.size == 0:
        return None
    vmin = max(np.nanpercentile(positive, pmin), 1e-30)
    vmax = max(np.nanpercentile(positive, pmax), vmin * 1.01)
    return LogNorm(vmin=vmin, vmax=vmax)


def replot_mollweide(arrays, replot_dir):
    required = {"los_l_deg", "los_b_deg"}
    if not required.issubset(arrays):
        print("Skip Mollweide: missing l/b arrays")
        return
    lon = np.deg2rad(arrays["los_l_deg"] - 180.0)
    lat = np.deg2rad(arrays["los_b_deg"])
    panels = [
        ("los_DM_total_pc_cm3", r"DM [pc cm$^{-3}$]", "viridis", "saved_DM_mollweide.png"),
        ("los_EM_ne2_total_pc_cm6", r"EM [pc cm$^{-6}$]", "magma", "saved_EM_mollweide.png"),
        ("los_EM_ne2_hot_pc_cm6", r"Hot EM [pc cm$^{-6}$]", "YlOrRd", "saved_hotEM_mollweide.png"),
    ]
    available = [p for p in panels if p[0] in arrays]
    if not available:
        print("Skip Mollweide: no DM/EM arrays")
        return
    fig = plt.figure(figsize=(10, 4.8 * len(available)), constrained_layout=True)
    for i, (key, label, cmap, outfile) in enumerate(available, start=1):
        ax = fig.add_subplot(len(available), 1, i, projection="mollweide")
        vals = arrays[key]
        sc = ax.scatter(lon, lat, c=vals, s=8, cmap=cmap, norm=positive_lognorm(vals))
        ax.grid(True, alpha=0.35)
        ax.set_title(key.replace("los_", ""))
        fig.colorbar(sc, ax=ax, orientation="horizontal", pad=0.08, label=label)
    out = replot_dir / "saved_DM_EM_hotEM_mollweide.png"
    fig.savefig(out, dpi=180)
    plt.show()
    print("Saved:", out)


def replot_hot_em_profile(arrays, metadata, replot_dir):
    if not {"los_angle_from_galactic_center_deg", "los_EM_ne2_hot_pc_cm6"}.issubset(arrays):
        print("Skip hot EM profile: missing angle or hot EM arrays")
        return
    angle = arrays["los_angle_from_galactic_center_deg"]
    hot_em = 1e3 * arrays["los_EM_ne2_hot_pc_cm6"]
    ok = np.isfinite(angle) & np.isfinite(hot_em) & (hot_em >= 0)
    angle = angle[ok]
    hot_em = hot_em[ok]
    fig, ax = plt.subplots(figsize=(8, 5.2), constrained_layout=True)
    ax.scatter(angle, hot_em, s=14, alpha=0.65, color="#d99a2b", edgecolor="none")
    if len(angle) >= 8:
        bins = np.linspace(0, 180, 19)
        centers = 0.5 * (bins[:-1] + bins[1:])
        med = np.full_like(centers, np.nan, dtype=float)
        lo = np.full_like(centers, np.nan, dtype=float)
        hi = np.full_like(centers, np.nan, dtype=float)
        for i in range(len(centers)):
            m = (angle >= bins[i]) & (angle < bins[i + 1])
            if np.count_nonzero(m):
                med[i] = np.nanmedian(hot_em[m])
                lo[i], hi[i] = np.nanpercentile(hot_em[m], [16, 84])
        good = np.isfinite(med)
        ax.plot(centers[good], med[good], color="black", lw=2, label="binned median")
        ax.fill_between(centers[good], lo[good], hi[good], color="black", alpha=0.18, linewidth=0, label="16-84%")
        ax.legend(fontsize=9)
    hotT = metadata.get("parameters", {}).get("HOT_TMIN_K", None)
    title = "Hot EM versus Galactic-centre angle" if hotT is None else f"Hot EM versus Galactic-centre angle (T >= {hotT:.1e} K)"
    ax.set_title(title)
    ax.set_xlabel("Angle from Galactic centre [deg]")
    ax.set_ylabel(r"Hot EM [$10^{-3}$ cm$^{-6}$ pc]")
    ax.grid(True, alpha=0.3)
    out = replot_dir / "saved_hot_EM_vs_GC_angle.png"
    fig.savefig(out, dpi=180)
    plt.show()
    print("Saved:", out)


def replot_projection_quantity(arrays, replot_dir, face_key, edge_key, face_extent_key, edge_extent_key, title, cmap, log10=True, cbar_label=""):
    if not {face_key, edge_key, face_extent_key, edge_extent_key}.issubset(arrays):
        print(f"Skip {title}: missing projection arrays")
        return
    fig, axes = plt.subplots(2, 1, figsize=(7, 10.5), constrained_layout=True)
    plot_arrays = []
    for key in [face_key, edge_key]:
        arr = arrays[key]
        if log10:
            arr = np.log10(np.where(arr > 0, arr, np.nan))
        plot_arrays.append(arr)
    finite = np.concatenate([a[np.isfinite(a)] for a in plot_arrays if np.any(np.isfinite(a))])
    vmin, vmax = (None, None) if finite.size == 0 else np.nanpercentile(finite, [2, 98])
    for ax, arr, ext_key, view in zip(axes, plot_arrays, [face_extent_key, edge_extent_key], ["face-on", "edge-on"]):
        im = ax.imshow(arr, origin="lower", extent=arrays[ext_key], cmap=cmap, vmin=vmin, vmax=vmax, interpolation="nearest")
        ax.set_title(f"{title} ({view})")
        ax.set_aspect("equal")
        ax.set_xlabel("kpc")
        ax.set_ylabel("kpc")
        fig.colorbar(im, ax=ax, label=cbar_label)
    out = replot_dir / ("saved_" + title.lower().replace(" ", "_").replace("/", "_") + "_face_edge.png")
    fig.savefig(out, dpi=180)
    plt.show()
    print("Saved:", out)


def replot_velocity_projection(arrays, replot_dir, quiver_step=None):
    required = {"projection_gas_face_sigma", "projection_gas_edge_sigma", "projection_gas_face_extent", "projection_gas_edge_extent", "projection_gas_face_vlos", "projection_gas_edge_vlos"}
    if not required.issubset(arrays):
        print("Skip velocity projection: missing velocity arrays")
        return
    if quiver_step is None:
        quiver_step = int(plot_metadata.get("parameters", {}).get("PROJECTION_QUIVER_STEP", 12)) if "plot_metadata" in globals() else 12
    fig, axes = plt.subplots(2, 1, figsize=(7, 10.5), constrained_layout=True)
    specs = [
        ("projection_gas_face", "face-on"),
        ("projection_gas_edge", "edge-on"),
    ]
    for ax, (prefix, view) in zip(axes, specs):
        sigma = arrays[f"{prefix}_sigma"]
        extent = arrays[f"{prefix}_extent"]
        bg = np.log10(np.where(sigma > 0, sigma, np.nan))
        vlos = arrays[f"{prefix}_vlos"]
        vmax = np.nanpercentile(np.abs(vlos[np.isfinite(vlos)]), 95) if np.any(np.isfinite(vlos)) else 1.0
        ax.imshow(bg, origin="lower", extent=extent, cmap="Greys", interpolation="nearest", alpha=0.45)
        im = ax.imshow(vlos, origin="lower", extent=extent, cmap="RdBu_r", norm=TwoSlopeNorm(vcenter=0, vmin=-vmax, vmax=vmax), interpolation="nearest", alpha=0.85)
        vx_key = f"{prefix}_vx"
        vy_key = f"{prefix}_vy"
        if vx_key in arrays and vy_key in arrays:
            ny, nx = arrays[vx_key].shape
            xs = np.linspace(extent[0], extent[1], nx)
            ys = np.linspace(extent[2], extent[3], ny)
            X, Y = np.meshgrid(xs, ys)
            sl = slice(None, None, quiver_step)
            ax.quiver(X[sl, sl], Y[sl, sl], arrays[vx_key][sl, sl], arrays[vy_key][sl, sl], **quiver_kwargs)
        ax.set_title(f"Gas velocity field ({view})")
        ax.set_aspect("equal")
        ax.set_xlabel("kpc")
        ax.set_ylabel("kpc")
        fig.colorbar(im, ax=ax, label="LOS velocity [km/s]")
    out = replot_dir / "saved_gas_velocity_face_edge.png"
    fig.savefig(out, dpi=180)
    plt.show()
    print("Saved:", out)


def replot_random_observer_boxplot(arrays, replot_dir):
    los_names = ["toward_center_in_plane", "anti_center_in_plane", "vertical_to_disk", "center", "anti_center", "vertical"]
    dm_keys = [k for k in arrays if k.startswith("random_observer_DM_total_pc_cm3")]
    if not dm_keys:
        print("Skip random observer boxplot: no random_observer arrays in package")
        return
    # Current saved random-observer arrays are column arrays, so use los_label_id if present.
    if "random_observer_los_label_id" not in arrays:
        print("Skip random observer boxplot: missing los_label_id")
        return
    label_id = arrays["random_observer_los_label_id"].astype(int)
    dm = arrays.get("random_observer_DM_total_pc_cm3")
    em = arrays.get("random_observer_EM_ne2_total_pc_cm6")
    if dm is None or em is None:
        print("Skip random observer boxplot: missing DM or EM")
        return
    labels = [los_names[i] if i < len(los_names) else str(i) for i in sorted(set(label_id))]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
    for ax, vals, ylabel, title in [(axes[0], dm, r"DM [pc cm$^{-3}$]", "Random observer DM"), (axes[1], em, r"EM [pc cm$^{-6}$]", "Random observer EM")]:
        data = [vals[label_id == i] for i in sorted(set(label_id))]
        ax.boxplot(data, tick_labels=labels, showfliers=False)
        for j, d in enumerate(data, start=1):
            jitter = np.linspace(-0.08, 0.08, len(d)) if len(d) else np.array([])
            ax.scatter(np.full(len(d), j) + jitter, d, s=20, alpha=0.7)
        ax.set_yscale("log")
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.grid(True, axis="y", alpha=0.3)
    out = replot_dir / "saved_random_observer_DM_EM_boxplot.png"
    fig.savefig(out, dpi=180)
    plt.show()
    print("Saved:", out)


plot_data_npz = MANUAL_PLOT_DATA_NPZ or None
plot_arrays, plot_metadata, replot_dir = load_saved_plot_data(plot_data_npz)

replot_mollweide(plot_arrays, replot_dir)
replot_hot_em_profile(plot_arrays, plot_metadata, replot_dir)
replot_projection_quantity(
    plot_arrays, replot_dir,
    "projection_gas_face_sigma", "projection_gas_edge_sigma",
    "projection_gas_face_extent", "projection_gas_edge_extent",
    "Gas surface density", "magma", True, "log gas surface density",
)
replot_projection_quantity(
    plot_arrays, replot_dir,
    "projection_star_face_sigma", "projection_star_edge_sigma",
    "projection_star_face_extent", "projection_star_edge_extent",
    "Stellar surface density", "inferno", True, "log stellar surface density",
)
replot_projection_quantity(
    plot_arrays, replot_dir,
    "projection_gas_face_temperature", "projection_gas_edge_temperature",
    "projection_gas_face_extent", "projection_gas_edge_extent",
    "Gas temperature", "turbo", True, "log T [K]",
)
replot_velocity_projection(plot_arrays, replot_dir)
replot_random_observer_boxplot(plot_arrays, replot_dir)


## Polished Replot From Saved Plot Data

This panel redraws saved plot-data products with larger figures, serif fonts, log-scaled hot-EM profiles, and auto-cropped projection panels. Use this instead of rerunning the expensive LOS calculation.


In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, TwoSlopeNorm

font = {"family": "serif", "size": 16}
font_label = {"family": "serif", "size": 20}
tick_size = 18
font_legend = {"family": "serif", "size": 16}
font_legend_small = {"family": "serif", "size": 13}

plt.rcParams.update({
    "font.family": "serif",
    "font.size": font["size"],
    "axes.titlesize": font_label["size"],
    "axes.labelsize": font_label["size"],
    "xtick.labelsize": tick_size,
    "ytick.labelsize": tick_size,
    "legend.fontsize": font_legend["size"],
    "figure.dpi": 130,
})

# Set this manually if you want a specific package. Otherwise the newest package is used.
MANUAL_PLOT_DATA_NPZ = None


def _find_latest_plot_data(outdir=OUTDIR):
    search_dirs = [Path(outdir) / "plot_data"]
    if "LEGACY_OUTPUT_BASE" in globals():
        search_dirs.append(LEGACY_OUTPUT_BASE / FOLDER_ALIASES[CODE] / "plot_data")
    search_dirs.extend([
        ROOT / "debug_outputs" / FOLDER_ALIASES.get(CODE, str(CODE)) / "plot_data",
        ROOT / "ARTI" / "MW_DM_EM_ARTI_output" / "plot_data",
    ])
    candidates = []
    for d in search_dirs:
        candidates.extend(Path(d).glob("*_plot_data.npz"))
    candidates = sorted(set(candidates), key=lambda p: p.stat().st_mtime)
    if not candidates:
        raise FileNotFoundError("No *_plot_data.npz found. Run the save plotting data cell first.")
    print("Using plot data:", candidates[-1])
    return candidates[-1]


def load_polished_plot_data(npz_path=None):
    npz_path = Path(npz_path) if npz_path is not None else _find_latest_plot_data()
    meta_path = npz_path.with_name(npz_path.name.replace("_plot_data.npz", "_plot_data_metadata.json"))
    arrays = dict(np.load(npz_path))
    metadata = json.loads(meta_path.read_text()) if meta_path.exists() else {}
    outdir = npz_path.parent / "replot_polished"
    outdir.mkdir(parents=True, exist_ok=True)
    print("Metadata:", meta_path if meta_path.exists() else "missing")
    print("Output:", outdir)
    return arrays, metadata, outdir


def _positive_lognorm(vals, pmin=2, pmax=98):
    vals = np.asarray(vals, dtype=float)
    pos = vals[np.isfinite(vals) & (vals > 0)]
    if pos.size == 0:
        return None
    vmin = max(np.nanpercentile(pos, pmin), 1e-30)
    vmax = max(np.nanpercentile(pos, pmax), vmin * 1.01)
    return LogNorm(vmin=vmin, vmax=vmax)


def _crop_slices_from_maps(arrs, pad_frac=0.18, min_pixels=20):
    mask = None
    for arr in arrs:
        if arr is None:
            continue
        m = np.isfinite(arr) & (arr != 0)
        mask = m if mask is None else (mask | m)
    if mask is None or not np.any(mask):
        return slice(None), slice(None)
    yy, xx = np.where(mask)
    y0, y1 = yy.min(), yy.max() + 1
    x0, x1 = xx.min(), xx.max() + 1
    ny, nx = mask.shape
    pad_x = max(int((x1 - x0) * pad_frac), min_pixels)
    pad_y = max(int((y1 - y0) * pad_frac), min_pixels)
    x0 = max(0, x0 - pad_x); x1 = min(nx, x1 + pad_x)
    y0 = max(0, y0 - pad_y); y1 = min(ny, y1 + pad_y)
    return slice(y0, y1), slice(x0, x1)


def _cropped_extent(extent, arr_shape, ys, xs):
    xmin, xmax, ymin, ymax = map(float, extent)
    ny, nx = arr_shape
    dx = (xmax - xmin) / nx
    dy = (ymax - ymin) / ny
    return [xmin + xs.start * dx, xmin + xs.stop * dx, ymin + ys.start * dy, ymin + ys.stop * dy]


def plot_polished_mollweide(arrays, outdir):
    if not {"los_l_deg", "los_b_deg"}.issubset(arrays):
        print("Skip Mollweide: missing l/b arrays")
        return
    lon = np.deg2rad(((arrays["los_l_deg"] + 180.0) % 360.0) - 180.0)
    lat = np.deg2rad(arrays["los_b_deg"])
    panels = [
        ("los_DM_total_pc_cm3", r"DM [pc cm$^{-3}$]", "viridis", "DM total"),
        ("los_EM_ne2_total_pc_cm6", r"EM [pc cm$^{-6}$]", "magma", "EM total"),
        ("los_EM_ne2_hot_pc_cm6", r"Hot EM [pc cm$^{-6}$]", "YlOrRd", "Hot EM"),
    ]
    panels = [p for p in panels if p[0] in arrays]
    if not panels:
        return
    fig, axes = plt.subplots(len(panels), 1, figsize=(13, 5.6 * len(panels)), subplot_kw={"projection": "mollweide"}, constrained_layout=True)
    axes = np.atleast_1d(axes)
    for ax, (key, label, cmap, title) in zip(axes, panels):
        vals = arrays[key]
        sc = ax.scatter(lon, lat, c=vals, s=10, cmap=cmap, norm=_positive_lognorm(vals), edgecolors="none", alpha=0.95, rasterized=True)
        ax.grid(True, alpha=0.3)
        ax.set_title(title, fontdict=font_label)
        cb = fig.colorbar(sc, ax=ax, orientation="horizontal", pad=0.08, label=label)
        cb.ax.tick_params(labelsize=tick_size)
    outfile = outdir / "polished_DM_EM_hotEM_mollweide.png"
    fig.savefig(outfile, dpi=220, bbox_inches="tight")
    plt.show()
    print("Saved:", outfile)


def plot_polished_hot_em_angle(arrays, metadata, outdir):
    need = {"los_angle_from_galactic_center_deg", "los_EM_ne2_hot_pc_cm6"}
    if not need.issubset(arrays):
        print("Skip hot EM profile: missing arrays")
        return
    angle = np.asarray(arrays["los_angle_from_galactic_center_deg"], dtype=float)
    hot_em = 1e3 * np.asarray(arrays["los_EM_ne2_hot_pc_cm6"], dtype=float)
    good = np.isfinite(angle) & np.isfinite(hot_em) & (hot_em > 0)
    angle = angle[good]
    hot_em = hot_em[good]
    fig, ax = plt.subplots(figsize=(11, 7), constrained_layout=True)
    ax.scatter(angle, hot_em, s=22, alpha=0.65, color="#d99a2b", edgecolor="none", label="LoS samples")
    if len(angle) >= 8:
        bins = np.linspace(0, 180, 19)
        centers = 0.5 * (bins[:-1] + bins[1:])
        med = np.full(len(centers), np.nan)
        lo = np.full(len(centers), np.nan)
        hi = np.full(len(centers), np.nan)
        for i in range(len(centers)):
            m = (angle >= bins[i]) & (angle < bins[i + 1])
            if np.count_nonzero(m):
                med[i] = np.nanmedian(hot_em[m])
                lo[i], hi[i] = np.nanpercentile(hot_em[m], [16, 84])
        ok = np.isfinite(med) & (med > 0)
        ax.plot(centers[ok], med[ok], color="black", lw=2.5, label="binned median")
        ax.fill_between(centers[ok], lo[ok], hi[ok], color="black", alpha=0.18, linewidth=0, label="16-84%")
    hotT = metadata.get("parameters", {}).get("HOT_TMIN_K", None)
    title = "Hot EM versus Galactic-centre angle" if hotT is None else rf"Hot EM versus Galactic-centre angle ($T \geq {hotT:.1e}$ K)"
    ax.set_yscale("log")
    ax.set_xlabel("Angle from Galactic centre [deg]", fontdict=font_label)
    ax.set_ylabel(r"Hot EM [$10^{-3}$ cm$^{-6}$ pc]", fontdict=font_label)
    ax.set_title(title, fontdict=font_label)
    ax.tick_params(axis="both", labelsize=tick_size)
    ax.grid(True, which="both", alpha=0.28)
    ax.legend(prop=font_legend)
    outfile = outdir / "polished_hot_EM_vs_GC_angle_logy.png"
    fig.savefig(outfile, dpi=220, bbox_inches="tight")
    plt.show()
    print("Saved:", outfile)


def plot_polished_projection(arrays, outdir, face_key, edge_key, face_ext_key, edge_ext_key, title, cmap, log10=True, cbar_label=""):
    if not {face_key, edge_key, face_ext_key, edge_ext_key}.issubset(arrays):
        print(f"Skip {title}: missing arrays")
        return
    face = np.asarray(arrays[face_key], dtype=float)
    edge = np.asarray(arrays[edge_key], dtype=float)
    face_plot = np.log10(np.where(face > 0, face, np.nan)) if log10 else face.copy()
    edge_plot = np.log10(np.where(edge > 0, edge, np.nan)) if log10 else edge.copy()
    ys_f, xs_f = _crop_slices_from_maps([face_plot])
    ys_e, xs_e = _crop_slices_from_maps([edge_plot])
    vals = np.concatenate([face_plot[np.isfinite(face_plot)], edge_plot[np.isfinite(edge_plot)]])
    vmin, vmax = (None, None) if vals.size == 0 else np.nanpercentile(vals, [2, 98])
    fig, axes = plt.subplots(2, 1, figsize=(9, 13), constrained_layout=True)
    for ax, arr, raw, ext, ys, xs, view in [
        (axes[0], face_plot, face, arrays[face_ext_key], ys_f, xs_f, "face-on"),
        (axes[1], edge_plot, edge, arrays[edge_ext_key], ys_e, xs_e, "edge-on"),
    ]:
        cropped = arr[ys, xs]
        crop_ext = _cropped_extent(ext, arr.shape, ys, xs)
        im = ax.imshow(cropped, origin="lower", extent=crop_ext, cmap=cmap, vmin=vmin, vmax=vmax, interpolation="nearest")
        ax.set_title(f"{title} ({view})", fontdict=font_label)
        ax.set_xlabel("kpc", fontdict=font_label)
        ax.set_ylabel("kpc", fontdict=font_label)
        ax.tick_params(axis="both", labelsize=tick_size)
        ax.set_aspect("equal")
        cb = fig.colorbar(im, ax=ax, label=cbar_label)
        cb.ax.tick_params(labelsize=tick_size)
        cb.set_label(cbar_label, fontdict=font_label)
    outfile = outdir / ("polished_" + title.lower().replace(" ", "_").replace("/", "_") + "_autozoom_face_edge.png")
    fig.savefig(outfile, dpi=220, bbox_inches="tight")
    plt.show()
    print("Saved:", outfile)


plot_arrays, plot_metadata, polished_outdir = load_polished_plot_data(MANUAL_PLOT_DATA_NPZ)
plot_polished_mollweide(plot_arrays, polished_outdir)
plot_polished_hot_em_angle(plot_arrays, plot_metadata, polished_outdir)
plot_polished_projection(
    plot_arrays, polished_outdir,
    "projection_gas_face_sigma", "projection_gas_edge_sigma",
    "projection_gas_face_extent", "projection_gas_edge_extent",
    "Gas surface density", "magma", True, "log surface density",
)
plot_polished_projection(
    plot_arrays, polished_outdir,
    "projection_star_face_sigma", "projection_star_edge_sigma",
    "projection_star_face_extent", "projection_star_edge_extent",
    "Stellar surface density", "inferno", True, "log surface density",
)
plot_polished_projection(
    plot_arrays, polished_outdir,
    "projection_gas_face_temperature", "projection_gas_edge_temperature",
    "projection_gas_face_extent", "projection_gas_edge_extent",
    "Gas temperature", "turbo", True, "log T [K]",
)


## Correct Saved Projection Velocities

Use this only for already-saved projection maps from older runs where the velocity maps were made in the simulation frame. It subtracts a known bulk velocity from the saved map-level `vx/vy/vlos` arrays. For GADGET3 results produced before the unit-base fix, rerun the pipeline instead of relying on this correction.


In [ ]:
import json
from pathlib import Path
import numpy as np

# Set this manually if you want to correct an old projection npz.
PROJECTION_NPZ_TO_CORRECT = OUTDIR / "projection_maps_face_edge.npz"
# Supply bulk velocity in the already face-on frame [vx, vy, vz] km/s.
# If the run metadata contains bulk_velocity_faceon_km_s, leave this as None.
MANUAL_BULK_VELOCITY_FACEON = None


def correct_saved_projection_velocity_maps(projection_npz=PROJECTION_NPZ_TO_CORRECT, bulk_faceon=MANUAL_BULK_VELOCITY_FACEON):
    projection_npz = Path(projection_npz)
    if not projection_npz.exists():
        raise FileNotFoundError(projection_npz)
    outdir = projection_npz.parent
    metadata_path = outdir / "MWlike_4pi_DM_EM_metadata.json"
    if bulk_faceon is None:
        if not metadata_path.exists():
            raise FileNotFoundError("No metadata with bulk velocity found. Set MANUAL_BULK_VELOCITY_FACEON.")
        metadata = json.loads(metadata_path.read_text())
        bulk_faceon = metadata.get("bulk_velocity_faceon_km_s")
        if bulk_faceon is None:
            raise ValueError("Metadata has no bulk_velocity_faceon_km_s. Set MANUAL_BULK_VELOCITY_FACEON.")
    bulk_faceon = np.asarray(bulk_faceon, dtype=float)
    data = dict(np.load(projection_npz))
    corrected = {k: v.copy() for k, v in data.items()}

    # face-on: plotted plane velocity = (vx, vy), LOS velocity = vz
    if "gas_face_vx" in corrected:
        corrected["gas_face_vx"] = corrected["gas_face_vx"] - bulk_faceon[0]
    if "gas_face_vy" in corrected:
        corrected["gas_face_vy"] = corrected["gas_face_vy"] - bulk_faceon[1]
    if "gas_face_vlos" in corrected:
        corrected["gas_face_vlos"] = corrected["gas_face_vlos"] - bulk_faceon[2]

    # edge-on: plotted plane velocity = (vx, vz), LOS velocity = vy
    if "gas_edge_vx" in corrected:
        corrected["gas_edge_vx"] = corrected["gas_edge_vx"] - bulk_faceon[0]
    if "gas_edge_vy" in corrected:
        corrected["gas_edge_vy"] = corrected["gas_edge_vy"] - bulk_faceon[2]
    if "gas_edge_vlos" in corrected:
        corrected["gas_edge_vlos"] = corrected["gas_edge_vlos"] - bulk_faceon[1]

    outfile = projection_npz.with_name(projection_npz.stem + "_bulk_corrected.npz")
    np.savez_compressed(outfile, **corrected)
    meta_out = outfile.with_name(outfile.stem + "_metadata.json")
    meta_out.write_text(json.dumps({
        "source_projection_npz": str(projection_npz),
        "bulk_velocity_faceon_km_s_subtracted": bulk_faceon.tolist(),
        "note": "Map-level velocity correction. Density/temperature maps unchanged.",
    }, indent=2))
    print("saved corrected projection maps:", outfile)
    print("saved correction metadata:", meta_out)
    return outfile

# Uncomment when you intentionally want to correct an old saved map.
# corrected_projection_npz = correct_saved_projection_velocity_maps()


## Synced Results Viewer Plot Panels

These cells mirror the lightweight viewer notebook panels. The shared plotting utilities above must be run first.


## Projection Maps For One Code

In [ ]:
if 'save_figure' not in globals() or 'load_json' not in globals():
    raise RuntimeError('Please run the Shared Results Plotting Utilities cell before this plotting panel.')

PLOT_CODE = SELECTED_CODE  # Change this to plot another code, e.g. 'GEAR', 'AREPO', 'G4Cal_Pablo'
PLOT_OUTDIR = OUTROOT / PLOT_CODE

font = {"family": "serif", "size": 16}
font_label = {"family": "serif", "size": 20}
tick_size = 18
font_legend = {"family": "serif", "size": 16}
font_legend_small = {"family": "serif", "size": 13}
PLOT_LIMIT_KPC = 15
FACE_EDGE_HEIGHT_RATIOS = [1.7, 1.0]
VELOCITY_QUIVER_STEP = 28
VELOCITY_QUIVER_SCALE = 120
VELOCITY_QUIVER_WIDTH = 0.0048
VELOCITY_QUIVER_ALPHA = 0.92

plt.rcParams.update({"font.family": font["family"], "font.size": font["size"]})

def positive_lognorm(arr, pmin=2, pmax=98):
    arr = np.asarray(arr, dtype=float)
    pos = arr[np.isfinite(arr) & (arr > 0)]
    if pos.size == 0:
        return None
    vmin = max(np.nanpercentile(pos, pmin), 1e-30)
    vmax = max(np.nanpercentile(pos, pmax), vmin * 1.01)
    return LogNorm(vmin=vmin, vmax=vmax)

def paired_positive_norm(data, keys, pmin=2, pmax=98):
    vals = []
    for key in keys:
        if key in data:
            arr = np.asarray(data[key], dtype=float)
            pos = arr[np.isfinite(arr) & (arr > 0)]
            if pos.size:
                vals.append(pos)
    if not vals:
        return None
    vals = np.concatenate(vals)
    vmin = max(np.nanpercentile(vals, pmin), 1e-30)
    vmax = max(np.nanpercentile(vals, pmax), vmin * 1.01)
    return LogNorm(vmin=vmin, vmax=vmax)

def paired_symmetric_norm(data, keys, pmax=98):
    vals = []
    for key in keys:
        if key in data:
            arr = np.asarray(data[key], dtype=float)
            finite = arr[np.isfinite(arr)]
            if finite.size:
                vals.append(finite)
    if not vals:
        return None
    vals = np.concatenate(vals)
    vmax = np.nanpercentile(np.abs(vals), pmax)
    if not np.isfinite(vmax) or vmax <= 0:
        return None
    from matplotlib.colors import TwoSlopeNorm
    return TwoSlopeNorm(vcenter=0.0, vmin=-vmax, vmax=vmax)

def add_velocity_quiver(ax, data, view, step=VELOCITY_QUIVER_STEP):
    vx_key, vy_key, ext_key = f'gas_{view}_vx', f'gas_{view}_vy', f'gas_{view}_extent'
    if vx_key not in data or vy_key not in data or ext_key not in data:
        return
    vx = np.asarray(data[vx_key], dtype=float)
    vy = np.asarray(data[vy_key], dtype=float)
    if vx.ndim != 2 or vy.shape != vx.shape:
        return
    extent = np.asarray(data[ext_key], dtype=float)
    ny, nx = vx.shape
    xs = np.linspace(extent[0], extent[1], nx)
    ys = np.linspace(extent[2], extent[3], ny)
    ix = np.arange(0, nx, step)
    iy = np.arange(0, ny, step)
    X, Y = np.meshgrid(xs[ix], ys[iy])
    U = vx[np.ix_(iy, ix)]
    V = vy[np.ix_(iy, ix)]
    keep = (np.isfinite(U) & np.isfinite(V)
            & (X >= -PLOT_LIMIT_KPC) & (X <= PLOT_LIMIT_KPC)
            & (Y >= -PLOT_LIMIT_KPC) & (Y <= PLOT_LIMIT_KPC))
    if np.any(keep):
        ax.quiver(
            X[keep], Y[keep], U[keep], V[keep], color='black', angles='xy',
            scale_units='xy', scale=VELOCITY_QUIVER_SCALE,
            width=VELOCITY_QUIVER_WIDTH, alpha=VELOCITY_QUIVER_ALPHA,
            headwidth=4.6, headlength=5.8, headaxislength=5.0, minlength=0.08, zorder=6,
        )

def style_map_axis(ax, row, col):
    ax.set_xlim(-PLOT_LIMIT_KPC, PLOT_LIMIT_KPC)
    ax.set_ylim(-PLOT_LIMIT_KPC, PLOT_LIMIT_KPC)
    ax.set_aspect('auto')  # keep subplot rows aligned with the requested 1.7:1 height ratio
    ax.tick_params(axis='both', labelsize=tick_size)
    if row == 1:
        ax.set_xlabel('X [kpc]', fontdict=font_label)
    else:
        ax.set_xticklabels([])
    if col == 0:
        ax.set_ylabel('Y [kpc]', fontdict=font_label)
    else:
        ax.set_yticklabels([])

def annotate_view(ax, text):
    ax.text(
        0.03, 0.94, text, transform=ax.transAxes, ha='left', va='top',
        color='white', fontdict=font_legend_small,
        bbox=dict(facecolor='black', alpha=0.45, edgecolor='none', pad=2),
    )

proj_npz = PLOT_OUTDIR / 'projection_maps_face_edge.npz'
if proj_npz.exists():
    data = np.load(proj_npz)
    specs = [
        dict(title='Gas', face_key='gas_face_sigma', edge_key='gas_edge_sigma', face_extent='gas_face_extent', edge_extent='gas_edge_extent', cmap='magma', norm_func=paired_positive_norm, cbar=r'$M_\odot$ kpc$^{-2}$'),
        dict(title='Stellar', face_key='star_face_sigma', edge_key='star_edge_sigma', face_extent='star_face_extent', edge_extent='star_edge_extent', cmap='inferno', norm_func=paired_positive_norm, cbar=r'$M_\odot$ kpc$^{-2}$'),
        dict(title='Velocity', face_key='gas_face_vlos', edge_key='gas_edge_vlos', face_extent='gas_face_extent', edge_extent='gas_edge_extent', cmap='RdBu_r', norm_func=paired_symmetric_norm, cbar=r'km s$^{-1}$', quiver=True),
        dict(title='Temperature', face_key='gas_face_temperature', edge_key='gas_edge_temperature', face_extent='gas_face_extent', edge_extent='gas_edge_extent', cmap='plasma', norm_func=paired_positive_norm, cbar='K'),
    ]
    fig, axes = plt.subplots(
        2, len(specs), figsize=(5.0 * len(specs), 8.2),
        gridspec_kw={'height_ratios': FACE_EDGE_HEIGHT_RATIOS},
        constrained_layout=True, squeeze=False,
    )
    for col, spec in enumerate(specs):
        norm = spec['norm_func'](data, [spec['face_key'], spec['edge_key']])
        last_im = None
        for row, (arr_key, ext_key, view) in enumerate([
            (spec['face_key'], spec['face_extent'], 'face-on'),
            (spec['edge_key'], spec['edge_extent'], 'edge-on'),
        ]):
            ax = axes[row, col]
            if arr_key not in data or ext_key not in data:
                ax.set_axis_off()
                continue
            last_im = ax.imshow(data[arr_key], origin='lower', extent=data[ext_key], cmap=spec['cmap'], norm=norm, interpolation='nearest')
            style_map_axis(ax, row, col)
            annotate_view(ax, view)
            if spec.get('quiver'):
                add_velocity_quiver(ax, data, 'face' if row == 0 else 'edge')
        if last_im is not None:
            cbar = fig.colorbar(last_im, ax=axes[:, col], shrink=0.88, pad=0.015)
            cbar.ax.tick_params(labelsize=font_legend_small['size'])
            cbar.set_label(f"{spec['title']} [{spec['cbar']}]", fontdict=font_legend)
    fig.suptitle(f'{code_label_with_redshift(PLOT_CODE)}: Projection maps', fontdict={'family': 'serif', 'size': 24}, y=0.995)
    save_figure(fig, f'{PLOT_CODE}_projection_one_code')
    plt.show()
else:
    print('Missing:', proj_npz)


## Projection Maps Across All Codes


In [ ]:
if 'save_figure' not in globals() or 'load_json' not in globals():
    raise RuntimeError('Please run the Shared Results Plotting Utilities cell before this plotting panel.')

# Across-code projection maps with selectable code list.
# Use "default"/"all"/None to plot every available valid code, or provide a list.
PLOT_CODE_LIST = globals().get('PLOT_CODE_LIST', 'default')
# Example:
# PLOT_CODE_LIST = ['ARTI', 'G4Cal_Pablo', 'GADGET3', 'GEAR']

def _load_projection_products(outroot=OUTROOT):
    products = {}
    for outdir in sorted(Path(outroot).glob('*')):
        if not is_production_output_dir(outdir):
            continue
        path = outdir / 'projection_maps_face_edge.npz'
        if path.exists():
            try:
                products[outdir.name] = np.load(path)
            except Exception as exc:
                print('Cannot load', path, repr(exc))
    return products

def _code_selection_or_all(products, selection=None):
    available = list(products.keys())
    lookup = {str(code).lower(): code for code in available}
    if selection is None:
        selection = globals().get('PLOT_CODE_LIST', 'default')
    if selection is None or selection == 'default' or selection == 'all':
        selected = available
    elif isinstance(selection, str):
        selected = [selection]
    else:
        selected = list(selection)
    selected = [lookup.get(str(code).lower(), code) for code in selected]
    selected = [code for code in selected if code in products]
    valid = []
    for code in selected:
        meta = load_json(OUTROOT / code / 'parallel_pipeline_metadata.json') or load_json(OUTROOT / code / 'MWlike_4pi_DM_EM_metadata.json')
        cfg = meta.get('config', {}) if meta else {}
        if code == 'CHANGA' and cfg.get('tipsy_length_unit_kpc') is None:
            print('Skipping CHANGA: Tipsy physical units are missing.')
            continue
        valid.append(code)
    missing = sorted(set(selected) - set(valid) - ({'CHANGA'} if 'CHANGA' in selected else set()))
    if missing:
        print('Requested codes missing projection products:', missing)
    return valid

def _global_positive_norm(products, keys, pmin=2, pmax=98):
    vals = []
    for data in products.values():
        for key in keys:
            if key in data:
                arr = np.asarray(data[key], dtype=float)
                pos = arr[np.isfinite(arr) & (arr > 0)]
                if pos.size:
                    vals.append(pos)
    if not vals:
        return None
    vals = np.concatenate(vals)
    vmin = max(np.nanpercentile(vals, pmin), 1e-30)
    vmax = max(np.nanpercentile(vals, pmax), vmin * 1.01)
    return LogNorm(vmin=vmin, vmax=vmax)

def _global_symmetric_norm(products, keys, pmax=98):
    vals = []
    for data in products.values():
        for key in keys:
            if key in data:
                arr = np.asarray(data[key], dtype=float)
                finite = arr[np.isfinite(arr)]
                if finite.size:
                    vals.append(finite)
    if not vals:
        return None
    vals = np.concatenate(vals)
    vmax = np.nanpercentile(np.abs(vals), pmax)
    if not np.isfinite(vmax) or vmax <= 0:
        return None
    from matplotlib.colors import TwoSlopeNorm
    return TwoSlopeNorm(vcenter=0.0, vmin=-vmax, vmax=vmax)

def _style_overview_axis(ax, row, col, n_rows, n_cols, view):
    ax.set_xlim(-PLOT_LIMIT_KPC, PLOT_LIMIT_KPC)
    ax.set_ylim(-PLOT_LIMIT_KPC, PLOT_LIMIT_KPC)
    ax.set_aspect('auto')
    ax.tick_params(axis='both', labelsize=tick_size)
    if row == n_rows - 1:
        ax.set_xlabel('X [kpc]', fontdict=font_label)
    else:
        ax.set_xticklabels([])
    if col == 0:
        ax.set_ylabel('Y [kpc]', fontdict=font_label)
    else:
        ax.set_yticklabels([])

def _projection_group_specs(products):
    return [
        dict(label=r'$\Sigma_{\rm gas}$', title='Gas', face_key='gas_face_sigma', edge_key='gas_edge_sigma', face_extent='gas_face_extent', edge_extent='gas_edge_extent', cmap='magma', norm=_global_positive_norm(products, ['gas_face_sigma', 'gas_edge_sigma']), cbar=r'$M_\odot$ kpc$^{-2}$'),
        dict(label=r'$\Sigma_{\star}$', title='Stellar', face_key='star_face_sigma', edge_key='star_edge_sigma', face_extent='star_face_extent', edge_extent='star_edge_extent', cmap='inferno', norm=_global_positive_norm(products, ['star_face_sigma', 'star_edge_sigma']), cbar=r'$M_\odot$ kpc$^{-2}$'),
        dict(label='Temperature', title='Temperature', face_key='gas_face_temperature', edge_key='gas_edge_temperature', face_extent='gas_face_extent', edge_extent='gas_edge_extent', cmap='plasma', norm=_global_positive_norm(products, ['gas_face_temperature', 'gas_edge_temperature']), cbar='K'),
        dict(label='Velocity', title='Velocity', face_key='gas_face_vlos', edge_key='gas_edge_vlos', face_extent='gas_face_extent', edge_extent='gas_edge_extent', cmap='RdBu_r', norm=_global_symmetric_norm(products, ['gas_face_vlos', 'gas_edge_vlos']), cbar=r'km s$^{-1}$', quiver=True),
    ]

def _plot_one_quantity_block(products, codes, spec):
    fig, axes = plt.subplots(
        2, len(codes), figsize=(3.35 * len(codes), 7.0),
        gridspec_kw={'height_ratios': FACE_EDGE_HEIGHT_RATIOS},
        constrained_layout=True, squeeze=False,
    )
    last_im = None
    for col, code in enumerate(codes):
        data = products[code]
        for row, (arr_key, ext_key, view) in enumerate([
            (spec['face_key'], spec['face_extent'], 'face-on'),
            (spec['edge_key'], spec['edge_extent'], 'edge-on'),
        ]):
            ax = axes[row, col]
            if arr_key not in data or ext_key not in data:
                ax.set_axis_off()
                continue
            last_im = ax.imshow(data[arr_key], origin='lower', extent=data[ext_key], cmap=spec['cmap'], norm=spec['norm'], interpolation='nearest')
            _style_overview_axis(ax, row, col, 2, len(codes), view)
            annotate_view(ax, view)
            if spec.get('quiver'):
                add_velocity_quiver(ax, data, 'face' if row == 0 else 'edge')
        axes[0, col].set_title(code_label_with_redshift(code), fontdict=font_label)
    if last_im is not None:
        cbar = fig.colorbar(last_im, ax=axes.ravel().tolist(), shrink=0.88, pad=0.01)
        cbar.set_label(f"{spec['label']} [{spec['cbar']}]", fontdict=font_legend)
        cbar.ax.tick_params(labelsize=font_legend_small['size'])
    fig.suptitle('Projection maps across codes', fontdict={'family': 'serif', 'size': 24})
    save_figure(fig, f"across_codes_{spec['title']}_projection")
    plt.show()

projection_products = _load_projection_products()
valid_codes = _code_selection_or_all(projection_products, PLOT_CODE_LIST)
if not valid_codes:
    print('No selected projection products found under', OUTROOT)
else:
    products = {code: projection_products[code] for code in valid_codes}
    for spec in _projection_group_specs(products):
        _plot_one_quantity_block(products, valid_codes, spec)


## Nested Projection Atlas Across Codes


In [ ]:
if 'save_figure' not in globals() or 'load_json' not in globals():
    raise RuntimeError('Please run the Shared Results Plotting Utilities cell before this plotting panel.')

# Compact nested atlas: columns are codes, rows are physical quantities.
# Each cell contains face-on on top and edge-on below.

projection_products = _load_projection_products()
atlas_codes = _code_selection_or_all(projection_products, PLOT_CODE_LIST)
if not atlas_codes:
    print('No selected projection products found under', OUTROOT)
else:
    atlas_products = {code: projection_products[code] for code in atlas_codes}
    atlas_specs = _projection_group_specs(atlas_products)
    n_rows = len(atlas_specs)
    n_cols = len(atlas_codes)

    fig = plt.figure(figsize=(3.15 * n_cols + 0.8, 4.75 * n_rows), constrained_layout=False)
    outer = fig.add_gridspec(
        n_rows, n_cols + 1,
        width_ratios=[1.0] * n_cols + [0.055],
        height_ratios=[1.0] * n_rows,
        left=0.085, right=0.945, bottom=0.06, top=0.935,
        wspace=0.055, hspace=0.18,
    )

    def _format_nested_axis(ax, row, col, i, n_rows, n_cols, view):
        ax.set_xlim(-PLOT_LIMIT_KPC, PLOT_LIMIT_KPC)
        ax.set_ylim(-PLOT_LIMIT_KPC, PLOT_LIMIT_KPC)
        if view == 'face-on':
            ax.set_box_aspect(1.0)
            ax.set_aspect('equal', adjustable='box')
        else:
            ax.set_box_aspect(0.45)
            ax.set_aspect('auto')
        ax.tick_params(axis='both', labelsize=tick_size, direction='out', pad=2)
        if not (row == n_rows - 1 and i == 1):
            ax.tick_params(labelbottom=False)
            ax.set_xlabel('')
        else:
            ax.set_xlabel('X [kpc]', fontdict=font_label, labelpad=3)
        if col != 0:
            ax.tick_params(labelleft=False)
            ax.set_ylabel('')
        else:
            ax.set_ylabel('Y [kpc]', fontdict=font_label, labelpad=3)

    for row, spec in enumerate(atlas_specs):
        row_axes = []
        last_im = None
        for col, code in enumerate(atlas_codes):
            data = atlas_products[code]
            sub = outer[row, col].subgridspec(
                2, 1,
                height_ratios=[1.0, 0.45],
                hspace=0.035,
            )
            face_ax = fig.add_subplot(sub[0, 0])
            edge_ax = fig.add_subplot(sub[1, 0])
            row_axes.extend([face_ax, edge_ax])
            for i, (ax, arr_key, ext_key, view) in enumerate([
                (face_ax, spec['face_key'], spec['face_extent'], 'face-on'),
                (edge_ax, spec['edge_key'], spec['edge_extent'], 'edge-on'),
            ]):
                if arr_key not in data or ext_key not in data:
                    ax.set_axis_off()
                    continue
                last_im = ax.imshow(
                    data[arr_key], origin='lower', extent=data[ext_key], cmap=spec['cmap'],
                    norm=spec['norm'], interpolation='nearest'
                )
                _format_nested_axis(ax, row, col, i, n_rows, n_cols, view)
                annotate_view(ax, view)
                if spec.get('quiver'):
                    add_velocity_quiver(ax, data, 'face' if i == 0 else 'edge')
            if row == 0:
                face_ax.set_title(code_label_with_redshift(code), fontdict=font_label, pad=8)
        if last_im is not None:
            cax = fig.add_subplot(outer[row, -1])
            cbar = fig.colorbar(last_im, cax=cax)
            cbar.set_label(f"{spec['label']} [{spec['cbar']}]", fontdict=font_legend)
            cbar.ax.tick_params(labelsize=font_legend_small['size'])
    fig.suptitle('Projection atlas across codes', fontdict={'family': 'serif', 'size': 26}, y=0.985)
    save_figure(fig, 'across_codes_nested_projection_atlas')
    plt.show()


## Mollweide And Hot-EM Diagnostic Panels


In [ ]:
if 'save_figure' not in globals() or 'load_json' not in globals():
    raise RuntimeError('Please run the Shared Results Plotting Utilities cell before this plotting panel.')

# Existing diagnostic products for one selected code.
# Set DIAGNOSTIC_CODE_OVERRIDE = 'GADGET3' (or another code) to force a code.
# Leave it as None to follow PLOT_CODE.
DIAGNOSTIC_CODE_OVERRIDE = globals().get('DIAGNOSTIC_CODE_OVERRIDE', None)
DIAGNOSTIC_CODE = DIAGNOSTIC_CODE_OVERRIDE or PLOT_CODE
DIAGNOSTIC_OUTDIR = OUTROOT / DIAGNOSTIC_CODE
DIAGNOSTIC_LABEL = code_label_with_redshift(DIAGNOSTIC_CODE)

mollweide_files = [
    ('DM', 'MWlike_4pi_DM_total_pc_cm3_mollweide.png'),
    ('Total EM', 'MWlike_4pi_EM_ne2_total_pc_cm6_mollweide.png'),
    ('Hot EM', 'MWlike_4pi_EM_ne2_hot_pc_cm6_mollweide.png'),
]
existing = [(label, DIAGNOSTIC_OUTDIR / fname) for label, fname in mollweide_files if (DIAGNOSTIC_OUTDIR / fname).exists()]
if not existing:
    print('No Mollweide PNGs found for', DIAGNOSTIC_CODE, 'under', DIAGNOSTIC_OUTDIR)
else:
    fig, axes = plt.subplots(len(existing), 1, figsize=(13, 4.6 * len(existing)), constrained_layout=True)
    axes = np.atleast_1d(axes)
    for ax, (label, path) in zip(axes, existing):
        ax.imshow(plt.imread(path))
        ax.set_title(f'{DIAGNOSTIC_LABEL}: {label} Mollweide', fontdict=font_label)
        ax.set_axis_off()
    save_figure(fig, f'{DIAGNOSTIC_CODE}_mollweide_diagnostics')
    plt.show()

def _plot_hot_em_vs_gc_angle_logx(ax, outdir, code_label):
    sightline_path = Path(outdir) / 'MWlike_4pi_DM_EM_sightlines.npz'
    if not sightline_path.exists():
        ax.set_axis_off()
        ax.text(0.5, 0.5, f'Missing {sightline_path.name}', transform=ax.transAxes, ha='center', va='center')
        return False
    with np.load(sightline_path) as data:
        if 'angle_from_galactic_center_deg' not in data.files or 'EM_ne2_hot_pc_cm6' not in data.files:
            ax.set_axis_off()
            ax.text(0.5, 0.5, 'Missing angle or hot-EM arrays', transform=ax.transAxes, ha='center', va='center')
            return False
        angle = np.asarray(data['angle_from_galactic_center_deg'], dtype=float)
        hot_em = np.asarray(data['EM_ne2_hot_pc_cm6'], dtype=float)
    mask = np.isfinite(angle) & np.isfinite(hot_em) & (angle > 0) & (hot_em >= 0)
    angle = angle[mask]
    hot_em = hot_em[mask]
    if angle.size == 0:
        ax.set_axis_off()
        ax.text(0.5, 0.5, 'No positive-angle hot-EM samples', transform=ax.transAxes, ha='center', va='center')
        return False
    ax.scatter(angle, hot_em, s=9, alpha=0.45, color='#d9951e', edgecolors='none', label='LOS samples')
    xmin = max(np.nanmin(angle), 0.1)
    xmax = max(np.nanmax(angle), xmin * 1.1)
    bins = np.logspace(np.log10(xmin), np.log10(xmax), 28)
    centers = np.sqrt(bins[:-1] * bins[1:])
    med = np.full_like(centers, np.nan, dtype=float)
    p16 = np.full_like(centers, np.nan, dtype=float)
    p84 = np.full_like(centers, np.nan, dtype=float)
    for i in range(len(centers)):
        in_bin = (angle >= bins[i]) & (angle < bins[i + 1])
        if np.count_nonzero(in_bin) >= 5:
            med[i] = np.nanmedian(hot_em[in_bin])
            p16[i], p84[i] = np.nanpercentile(hot_em[in_bin], [16, 84])
    ok = np.isfinite(med)
    if np.any(ok):
        ax.fill_between(centers[ok], p16[ok], p84[ok], color='0.7', alpha=0.5, lw=0, label='16-84%')
        ax.plot(centers[ok], med[ok], color='black', lw=2.3, label='binned median')
    ax.set_xscale('log')
    ax.set_xlim(xmin, xmax)
    ax.set_xlabel('Angle from Galactic centre [deg]', fontdict=font_label)
    ax.set_ylabel(r'Hot EM [pc cm$^{-6}$]', fontdict=font_label)
    ax.set_title(f'{code_label}: Hot EM vs GC angle', fontdict=font_label)
    ax.tick_params(axis='both', labelsize=tick_size)
    ax.grid(True, which='both', alpha=0.25)
    ax.legend(frameon=True, fontsize=font_legend_small['size'])
    return True

polar_path = DIAGNOSTIC_OUTDIR / 'MWlike_4pi_hot_EM_GC_polar.png'
if not polar_path.exists() and not (DIAGNOSTIC_OUTDIR / 'MWlike_4pi_DM_EM_sightlines.npz').exists():
    print('No hot-EM diagnostic products found for', DIAGNOSTIC_CODE, 'under', DIAGNOSTIC_OUTDIR)
else:
    fig, axes = plt.subplots(1, 2, figsize=(14.5, 6.4), constrained_layout=True)
    if polar_path.exists():
        axes[0].imshow(plt.imread(polar_path))
        axes[0].set_title(f'{DIAGNOSTIC_LABEL}: Hot EM GC polar', fontdict=font_label)
        axes[0].set_axis_off()
    else:
        axes[0].set_axis_off()
        axes[0].text(0.5, 0.5, 'Missing polar PNG', transform=axes[0].transAxes, ha='center', va='center')
    _plot_hot_em_vs_gc_angle_logx(axes[1], DIAGNOSTIC_OUTDIR, DIAGNOSTIC_LABEL)
    save_figure(fig, f'{DIAGNOSTIC_CODE}_hot_em_diagnostics')
    plt.show()


## DM PDF Parametric Fits Across Codes


In [ ]:
if 'save_figure' not in globals() or 'load_json' not in globals():
    raise RuntimeError('Please run the Shared Results Plotting Utilities cell before this plotting panel.')

# DM PDF data + parametric fits. This is intentionally DM-only for clarity.
from scipy import stats, special

def load_dm_fit_distributions(outroot=OUTROOT):
    per_code = {}
    for npz_path in sorted(Path(outroot).glob('*/MWlike_4pi_DM_EM_sightlines.npz')):
        code_name = npz_path.parent.name
        if not is_production_output_name(code_name):
            continue
        with np.load(npz_path) as data:
            if 'DM_total_pc_cm3' not in data.files:
                continue
            vals = np.asarray(data['DM_total_pc_cm3'], dtype=float)
            vals = vals[np.isfinite(vals) & (vals > 0)]
            if vals.size > 5:
                per_code[code_name] = {'DM_total_pc_cm3': vals}
    return per_code


def fit_log10_gmm(y, n_components=3, max_iter=300, tol=1e-7):
    """Fit a 1D Gaussian mixture to log10 values with a small EM implementation."""
    y = np.asarray(y, dtype=float)
    y = y[np.isfinite(y)]
    if y.size < max(20, 5 * n_components):
        raise ValueError('not enough samples for GMM')
    qs = np.linspace(10, 90, n_components)
    means = np.nanpercentile(y, qs)
    sigma0 = np.nanstd(y)
    if not np.isfinite(sigma0) or sigma0 <= 0:
        raise ValueError('zero scatter in log10 data')
    sigmas = np.full(n_components, max(sigma0 / n_components, 1e-3), dtype=float)
    weights = np.full(n_components, 1.0 / n_components, dtype=float)
    prev_ll = -np.inf
    for _ in range(max_iter):
        log_resp = np.column_stack([
            np.log(max(weights[k], 1e-300)) + stats.norm.logpdf(y, loc=means[k], scale=max(sigmas[k], 1e-6))
            for k in range(n_components)
        ])
        log_norm = special.logsumexp(log_resp, axis=1)
        ll = float(np.sum(log_norm))
        resp = np.exp(log_resp - log_norm[:, None])
        nk = resp.sum(axis=0) + 1e-300
        weights = nk / y.size
        means = (resp * y[:, None]).sum(axis=0) / nk
        var = (resp * (y[:, None] - means) ** 2).sum(axis=0) / nk
        sigmas = np.sqrt(np.maximum(var, 1e-6))
        if np.isfinite(prev_ll) and abs(ll - prev_ll) < tol * (abs(prev_ll) + 1.0):
            break
        prev_ll = ll
    order = np.argsort(means)
    return weights[order], means[order], sigmas[order], ll

def log10_gmm_pdf_x(x, weights, means, sigmas):
    """Convert a Gaussian mixture PDF in y=log10(x) to a PDF in x."""
    x = np.asarray(x, dtype=float)
    y = np.log10(x)
    pdf_y = np.zeros_like(x, dtype=float)
    for w, mu, sigma in zip(weights, means, sigmas):
        pdf_y += w * stats.norm.pdf(y, loc=mu, scale=max(sigma, 1e-6))
    return pdf_y / (x * np.log(10.0))

DM_FIT_CODE_LIST = globals().get('DM_FIT_CODE_LIST', PLOT_CODE_LIST)
all_dist = load_dm_fit_distributions()
if 'projection_products' in globals():
    fit_codes = _code_selection_or_all({code: None for code in all_dist}, DM_FIT_CODE_LIST)
else:
    fit_codes = list(all_dist)
fit_codes = [code for code in fit_codes if code in all_dist and 'DM_total_pc_cm3' in all_dist[code]]

if not fit_codes:
    print('No DM distributions found for selected codes.')
else:
    ncols = min(3, len(fit_codes))
    nrows = int(np.ceil(len(fit_codes) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(6.0 * ncols, 4.7 * nrows), constrained_layout=True)
    axes = np.atleast_1d(axes).ravel()
    fit_rows = []
    for ax, code in zip(axes, fit_codes):
        vals = np.asarray(all_dist[code]['DM_total_pc_cm3'], dtype=float)
        vals = vals[np.isfinite(vals) & (vals > 0)]
        xmin = max(np.nanpercentile(vals, 0.2), np.nanmin(vals))
        xmax = np.nanpercentile(vals, 99.8)
        bins = np.logspace(np.log10(xmin), np.log10(xmax), 52)
        xgrid = np.logspace(np.log10(xmin), np.log10(xmax), 500)
        ax.hist(vals, bins=bins, density=True, histtype='step', lw=2.1, color='black', label=f'data (N={len(vals)})')

        try:
            shape, loc, scale = stats.lognorm.fit(vals, floc=0)
            ax.plot(xgrid, stats.lognorm.pdf(xgrid, shape, loc=loc, scale=scale), lw=2, label='Log-normal')
            fit_rows.append({'code': code, 'model': 'lognormal', 'shape': shape, 'loc': loc, 'scale': scale})
        except Exception as exc:
            print(code, 'lognormal fit failed:', repr(exc))
        try:
            weights, means, sigmas, loglike = fit_log10_gmm(np.log10(vals), n_components=3)
            gmm_pdf = log10_gmm_pdf_x(xgrid, weights, means, sigmas)
            ax.plot(xgrid, gmm_pdf, lw=2, ls='-.', label='GMM-3 log10')
            fit_rows.append({
                'code': code,
                'model': 'gmm3_log10',
                'weights': ';'.join(f'{v:.8g}' for v in weights),
                'means_log10': ';'.join(f'{v:.8g}' for v in means),
                'sigmas_log10': ';'.join(f'{v:.8g}' for v in sigmas),
                'loglike': loglike,
            })
        except Exception as exc:
            print(code, 'GMM-3 log10 fit failed:', repr(exc))
        try:
            c, loc, scale = stats.weibull_min.fit(vals, floc=0)
            ax.plot(xgrid, stats.weibull_min.pdf(xgrid, c, loc=loc, scale=scale), lw=2, ls='--', label='Weibull')
            fit_rows.append({'code': code, 'model': 'weibull', 'shape': c, 'loc': loc, 'scale': scale})
        except Exception as exc:
            print(code, 'weibull fit failed:', repr(exc))
        try:
            threshold = np.nanpercentile(vals, 90)
            tail = vals[vals >= threshold] - threshold
            gpd_c, gpd_loc, gpd_scale = stats.genpareto.fit(tail, floc=0)
            tail_pdf = np.zeros_like(xgrid)
            mask = xgrid >= threshold
            tail_frac = tail.size / vals.size
            tail_pdf[mask] = tail_frac * stats.genpareto.pdf(xgrid[mask] - threshold, gpd_c, loc=gpd_loc, scale=gpd_scale)
            ax.plot(xgrid[mask], tail_pdf[mask], lw=2, ls=':', label='GPD tail')
            fit_rows.append({'code': code, 'model': 'gpd_tail_p90', 'shape': gpd_c, 'loc': gpd_loc, 'scale': gpd_scale, 'threshold': threshold})
        except Exception as exc:
            print(code, 'GPD tail fit failed:', repr(exc))

        ax.set_xscale('log')
        ax.set_yscale('log')
        ax.set_title(code, fontdict=font_label)
        ax.set_xlabel(r'DM [pc cm$^{-3}$]', fontdict=font_label)
        ax.set_ylabel('PDF', fontdict=font_label)
        ax.tick_params(axis='both', labelsize=tick_size)
        ax.grid(True, which='both', alpha=0.25)
        ax.legend(frameon=False, fontsize=font_legend_small['size'])
    for ax in axes[len(fit_codes):]:
        ax.set_axis_off()
    save_figure(fig, 'DM_pdf_parametric_fits')
    plt.show()

    if fit_rows:
        try:
            import pandas as pd
            fit_table = pd.DataFrame(fit_rows)
            fit_table.to_csv(FIGURE_DIR / 'DM_pdf_parametric_fit_parameters.csv', index=False)
            display(fit_table)
        except Exception:
            print(fit_rows)


## LOS PDF Histograms Across Codes

In [ ]:
FIT_QUANTITIES = {
    'DM_total_pc_cm3': r'DM [pc cm$^{-3}$]',
    'EM_ne2_total_pc_cm6': r'EM [pc cm$^{-6}$]',
    'EM_ne2_hot_pc_cm6': r'Hot EM [pc cm$^{-6}$]',
}
PDF_XMIN_BY_QUANTITY = {
    'DM_total_pc_cm3': 10,
    'EM_ne2_total_pc_cm6': 1e-2,
    'EM_ne2_hot_pc_cm6': 10**-2.5,
}
PDF_LEGEND_SIZE = max(8, font_legend_small['size'] - 2)

def load_per_code_distributions(outroot=OUTROOT):
    per_code = {}
    for npz_path in sorted(Path(outroot).glob('*/MWlike_4pi_DM_EM_sightlines.npz')):
        code_name = npz_path.parent.name
        if not is_production_output_name(code_name):
            continue
        with np.load(npz_path) as data:
            arrays = {}
            for key in FIT_QUANTITIES:
                if key in data.files:
                    vals = np.asarray(data[key], dtype=float)
                    vals = vals[np.isfinite(vals) & (vals > 0)]
                    if vals.size > 5:
                        arrays[key] = vals
        if arrays:
            per_code[code_name] = arrays
    return per_code

per_code = load_per_code_distributions()
quantities = [q for q in FIT_QUANTITIES if any(q in v for v in per_code.values())]
if quantities:
    colors = plt.cm.tab10(np.linspace(0, 1, max(3, len(per_code))))
    code_colors = {code: colors[i] for i, code in enumerate(sorted(per_code))}
    fig, axes = plt.subplots(len(quantities), 1, figsize=(10, 4.5 * len(quantities)), constrained_layout=True)
    axes = np.atleast_1d(axes)
    for ax, quantity in zip(axes, quantities):
        all_vals = np.concatenate([v[quantity] for v in per_code.values() if quantity in v])
        xmin = PDF_XMIN_BY_QUANTITY.get(quantity, 1e-2)
        all_vals = all_vals[np.isfinite(all_vals) & (all_vals >= xmin)]
        if all_vals.size == 0:
            ax.set_axis_off()
            ax.text(0.5, 0.5, f'No {quantity} values >= {xmin:g}', transform=ax.transAxes, ha='center', va='center')
            continue
        xmax = max(np.nanpercentile(all_vals, 99.8), xmin * 1.1)
        bins = np.logspace(np.log10(xmin), np.log10(xmax), 56)
        for code, arrays in sorted(per_code.items()):
            if quantity not in arrays:
                continue
            vals = arrays[quantity]
            vals = vals[np.isfinite(vals) & (vals >= xmin)]
            if vals.size == 0:
                continue
            ax.hist(
                vals, bins=bins, density=True, histtype='step', lw=2,
                color=code_colors[code], label=f'{code_label_with_redshift(code)} (N={len(vals)})'
            )
        ax.set_xscale('log')
        ax.set_yscale('log')
        ax.set_xlim(xmin, xmax)
        ax.set_xlabel(FIT_QUANTITIES[quantity], fontdict=font_label)
        ax.set_ylabel('PDF', fontdict=font_label)
        ax.tick_params(axis='both', labelsize=tick_size)
        ax.grid(True, which='both', alpha=0.25)
        ax.legend(ncol=2, frameon=False, fontsize=PDF_LEGEND_SIZE)
    save_figure(fig, 'LOS_pdf_histograms_across_codes')
    plt.show()
else:
    print('No LOS PDF quantities found under', OUTROOT)


## DM and EM Component Contributions


In [ ]:
# DM/EM component contribution distributions for one selected code.
# Change COMPONENT_CODE to inspect another simulation output.
COMPONENT_CODE = globals().get('COMPONENT_CODE', globals().get('PLOT_CODE', SELECTED_CODE))
COMPONENT_OUTDIR = OUTROOT / COMPONENT_CODE
component_npz = COMPONENT_OUTDIR / 'MWlike_4pi_DM_EM_sightlines.npz'

component_specs = [
    (
        'DM component distribution',
        r'DM [pc cm$^{-3}$]',
        [
            ('Total', 'DM_total_pc_cm3'),
            ('ISM', 'DM_ISM_pc_cm3'),
            ('CGM', 'DM_CGM_pc_cm3'),
            ('Hot', 'DM_hot_pc_cm3'),
        ],
    ),
    (
        'EM component distribution',
        r'EM [pc cm$^{-6}$]',
        [
            ('Total', 'EM_ne2_total_pc_cm6'),
            ('ISM', 'EM_ne2_ISM_pc_cm6'),
            ('CGM', 'EM_ne2_CGM_pc_cm6'),
            ('Hot', 'EM_ne2_hot_pc_cm6'),
        ],
    ),
]

def _component_array(data, key):
    if key not in data.files:
        return np.array([], dtype=float)
    vals = np.asarray(data[key], dtype=float)
    return vals[np.isfinite(vals) & (vals > 0)]

if not component_npz.exists():
    print('Missing:', component_npz)
else:
    with np.load(component_npz) as data:
        fig, axes = plt.subplots(1, 2, figsize=(13.0, 5.2), constrained_layout=True)
        rng = np.random.default_rng(12345)
        for ax, (title, ylabel, entries) in zip(axes, component_specs):
            labels = []
            arrays = []
            for label, key in entries:
                vals = _component_array(data, key)
                if vals.size == 0:
                    print(f'{COMPONENT_CODE}: missing or empty {key}')
                labels.append(label)
                arrays.append(vals if vals.size else np.array([np.nan]))

            ax.boxplot(arrays, labels=labels, showfliers=False)
            for i, vals in enumerate(arrays, start=1):
                vals = vals[np.isfinite(vals) & (vals > 0)]
                if vals.size == 0:
                    continue
                jitter = rng.uniform(-0.08, 0.08, vals.size)
                ax.scatter(np.full(vals.size, i) + jitter, vals, s=7, alpha=0.35, rasterized=True)

            ax.set_yscale('log')
            ax.set_ylabel(ylabel, fontdict=font_label if 'font_label' in globals() else None)
            ax.set_title(title, fontdict=font_label if 'font_label' in globals() else None)
            ax.tick_params(axis='both', labelsize=tick_size if 'tick_size' in globals() else None)
            ax.grid(True, axis='y', which='both', alpha=0.28)
        fig.suptitle(f'{code_label_with_redshift(COMPONENT_CODE)}: DM/EM component contributions', fontdict={'family': 'serif', 'size': 22})
        save_figure(fig, f'{COMPONENT_CODE}_DM_EM_component_distributions')
        plt.show()


## DM and EM Component Contributions Across Selected Codes


In [ ]:
# DM/EM component contribution distributions for all selected codes.
# Use PLOT_CODE_LIST, or set COMPONENT_CODE_LIST manually.  
#COMPONENT_CODE_LIST = ['ARTI', 'Enzo', 'G4Cal_Pablo', 'GADGET3', 'GEAR']
#COMPONENT_CODE_LIST = 'all'
COMPONENT_CODE_LIST = globals().get('COMPONENT_CODE_LIST', globals().get('PLOT_CODE_LIST', 'default'))

def _available_component_codes(outroot=OUTROOT):
    return sorted(p.parent.name for p in Path(outroot).glob('*/MWlike_4pi_DM_EM_sightlines.npz') if is_production_output_name(p.parent.name))

def _select_component_codes(selection=COMPONENT_CODE_LIST):
    available = _available_component_codes()
    lookup = {str(code).lower(): code for code in available}
    if selection is None or selection == 'default' or selection == 'all':
        selected = available
    elif isinstance(selection, str):
        selected = [selection]
    else:
        selected = list(selection)
    selected = [lookup.get(str(code).lower(), code) for code in selected]
    selected = [code for code in selected if code in available]
    if not selected:
        print('No selected component files found. Available:', available)
    return selected

component_specs = [
    (
        'DM component distribution',
        r'DM [pc cm$^{-3}$]',
        [
            ('Total', 'DM_total_pc_cm3'),
            ('ISM', 'DM_ISM_pc_cm3'),
            ('CGM', 'DM_CGM_pc_cm3'),
            ('Hot', 'DM_hot_pc_cm3'),
        ],
    ),
    (
        'EM component distribution',
        r'EM [pc cm$^{-6}$]',
        [
            ('Total', 'EM_ne2_total_pc_cm6'),
            ('ISM', 'EM_ne2_ISM_pc_cm6'),
            ('CGM', 'EM_ne2_CGM_pc_cm6'),
            ('Hot', 'EM_ne2_hot_pc_cm6'),
        ],
    ),
]

def _component_array(data, key):
    if key not in data.files:
        return np.array([], dtype=float)
    vals = np.asarray(data[key], dtype=float)
    return vals[np.isfinite(vals) & (vals > 0)]

component_codes = _select_component_codes()

for component_code in component_codes:
    component_npz = OUTROOT / component_code / 'MWlike_4pi_DM_EM_sightlines.npz'
    with np.load(component_npz) as data:
        fig, axes = plt.subplots(1, 2, figsize=(13.0, 5.2), constrained_layout=True)
        rng = np.random.default_rng(12345)

        for ax, (title, ylabel, entries) in zip(axes, component_specs):
            labels = []
            arrays = []

            for label, key in entries:
                vals = _component_array(data, key)
                labels.append(label)
                arrays.append(vals if vals.size else np.array([np.nan]))

            ax.boxplot(arrays, labels=labels, showfliers=False)

            for i, vals in enumerate(arrays, start=1):
                vals = vals[np.isfinite(vals) & (vals > 0)]
                if vals.size == 0:
                    continue
                jitter = rng.uniform(-0.08, 0.08, vals.size)
                ax.scatter(
                    np.full(vals.size, i) + jitter,
                    vals,
                    s=7,
                    alpha=0.35,
                    rasterized=True,
                )

            ax.set_yscale('log')
            ax.set_ylabel(ylabel, fontdict=font_label)
            ax.set_title(title, fontdict=font_label)
            ax.tick_params(axis='both', labelsize=tick_size)
            ax.grid(True, axis='y', which='both', alpha=0.28)

        fig.suptitle(
            f'{code_label_with_redshift(component_code)}: DM/EM component contributions',
            fontdict={'family': 'serif', 'size': 22},
        )
        save_figure(fig, f'{component_code}_DM_EM_component_distributions')
        plt.show()

## Random Observer Special LOS

In [ ]:
# Plot random-observer special LOS results.
# Default follows the current PLOT_CODE. Set RANDOM_OBSERVER_CODE_LIST = 'all' or a list of codes to compare multiple codes.
RANDOM_OBSERVER_CODE_LIST = globals().get('RANDOM_OBSERVER_CODE_LIST', [PLOT_CODE])
RANDOM_OBSERVER_YMIN = 1e-2

RANDOM_OBSERVER_QUANTITIES = [
    ('DM_total_pc_cm3', r'DM [pc cm$^{-3}$]'),
    ('EM_ne2_total_pc_cm6', r'EM [pc cm$^{-6}$]'),
    ('EM_ne2_hot_pc_cm6', r'Hot EM [pc cm$^{-6}$]'),
]
LOS_TYPE_ORDER = ['toward_center_in_plane', 'anti_center_in_plane', 'vertical_to_disk']
LOS_TYPE_LABELS = {
    'toward_center_in_plane': 'Toward centre',
    'anti_center_in_plane': 'Anti-centre',
    'vertical_to_disk': 'Vertical',
}

def _available_random_observer_codes(outroot=OUTROOT):
    return sorted(p.parent.name for p in Path(outroot).glob('*/random_observer_special_los_DM_EM.csv') if is_production_output_name(p.parent.name))

def _select_random_observer_codes(selection=RANDOM_OBSERVER_CODE_LIST):
    available = _available_random_observer_codes()
    lookup = {str(code).lower(): code for code in available}
    if selection is None or selection == 'default':
        selected = [PLOT_CODE]
    elif selection == 'all':
        selected = available
    elif isinstance(selection, str):
        selected = [selection]
    else:
        selected = list(selection)
    selected = [lookup.get(str(code).lower(), code) for code in selected]
    selected = [code for code in selected if code in available]
    if not selected:
        print('No selected random-observer CSV files found. Available:', available)
    return selected

def _load_random_observer_rows(code):
    path = OUTROOT / code / 'random_observer_special_los_DM_EM.csv'
    rows = []
    with path.open(newline='') as f:
        reader = csv.DictReader(f)
        for row in reader:
            parsed = {'code': code, 'los_type': row.get('los_type', '')}
            for key, val in row.items():
                if key in {'los_type'}:
                    continue
                try:
                    parsed[key] = float(val)
                except Exception:
                    parsed[key] = val
            rows.append(parsed)
    return rows

random_codes = _select_random_observer_codes()
random_rows = []
for code in random_codes:
    random_rows.extend(_load_random_observer_rows(code))

if not random_rows:
    print('No random-observer data to plot.')
else:
    n_rows = len(RANDOM_OBSERVER_QUANTITIES)
    n_cols = len(LOS_TYPE_ORDER)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.3 * n_cols, 3.75 * n_rows), constrained_layout=True, squeeze=False)
    code_colors = {code: plt.cm.tab10(i % 10) for i, code in enumerate(random_codes)}
    rng = np.random.default_rng(12345)

    for r, (quantity, ylabel) in enumerate(RANDOM_OBSERVER_QUANTITIES):
        row_vals = []
        for c, los_type in enumerate(LOS_TYPE_ORDER):
            ax = axes[r, c]
            positions = np.arange(1, len(random_codes) + 1)
            box_data = []
            used_positions = []
            used_labels = []
            for pos, code in zip(positions, random_codes):
                vals = np.asarray([
                    row.get(quantity, np.nan) for row in random_rows
                    if row.get('code') == code and row.get('los_type') == los_type
                ], dtype=float)
                vals = vals[np.isfinite(vals) & (vals > 0)]
                vals = vals[vals >= RANDOM_OBSERVER_YMIN]
                if vals.size == 0:
                    continue
                box_data.append(vals)
                used_positions.append(pos)
                used_labels.append(code_label_with_redshift(code))
                jitter = rng.normal(0.0, 0.035, size=vals.size)
                ax.scatter(np.full(vals.size, pos) + jitter, vals, s=13, alpha=0.45, color=code_colors[code], edgecolors='none')
                row_vals.append(vals)
            if box_data:
                bp = ax.boxplot(
                    box_data, positions=used_positions, widths=0.52,
                    showfliers=False, patch_artist=True,
                    medianprops={'color': 'black', 'lw': 1.4},
                    boxprops={'lw': 1.2}, whiskerprops={'lw': 1.1}, capprops={'lw': 1.1},
                )
                for patch, pos in zip(bp['boxes'], used_positions):
                    code = random_codes[int(pos) - 1]
                    patch.set_facecolor(code_colors[code])
                    patch.set_alpha(0.28)
            ax.set_yscale('log')
            ax.set_ylim(bottom=RANDOM_OBSERVER_YMIN)
            ax.set_xticks(used_positions)
            ax.set_xticklabels(used_labels, rotation=28, ha='right', fontsize=max(8, font_legend_small['size'] - 2))
            if r == 0:
                ax.set_title(LOS_TYPE_LABELS.get(los_type, los_type), fontdict=font_label)
            if c == 0:
                ax.set_ylabel(ylabel, fontdict=font_label)
            ax.tick_params(axis='y', labelsize=tick_size)
            ax.grid(True, which='both', axis='y', alpha=0.25)
        if row_vals:
            ymax = max(np.nanpercentile(np.concatenate(row_vals), 99.5), RANDOM_OBSERVER_YMIN * 10)
            for c in range(n_cols):
                axes[r, c].set_ylim(RANDOM_OBSERVER_YMIN, ymax * 1.4)
    fig.suptitle('Random observer special LOS', fontdict={'family': 'serif', 'size': 24}, y=1.01)
    suffix = 'all' if RANDOM_OBSERVER_CODE_LIST == 'all' else '_'.join(random_codes)
    save_figure(fig, f'random_observer_special_los_{safe_filename(suffix)}')
    plt.show()

    try:
        import pandas as pd
        display(pd.DataFrame(random_rows).head())
    except Exception:
        pass


## Quick Dataset Structure Inspection

Use this lightweight panel to inspect each code's yt-visible data structure, units, particle types, and key gas/star fields before running expensive LOS calculations.


In [ ]:
from pathlib import Path
import sys
import importlib
import json
import csv
import numpy as np

# This notebook should be stored in the z0 directory.
ROOT = Path('/home/zhaozhang/local/AGORA_work/AGORA_Data/z0')
PIPELINE = ROOT / 'AGORA_parallel_DM_EM_projection_pipeline.py'

if not PIPELINE.exists():
    raise FileNotFoundError(f'Missing pipeline script: {PIPELINE}')

sys.path.insert(0, str(ROOT))
import AGORA_parallel_DM_EM_projection_pipeline as agora
agora = importlib.reload(agora)

# Quick structure inspection for AGORA z0 datasets.
# Edit this list if you only want a subset, e.g. ["Enzo", "G4Cal_Pablo"].

INSPECT_CODES = ["ARTI", "Enzo", "AREPO", "GADGET3", "GEAR", "CHANGA", "G4Cal_Pablo"]
INSPECT_MAX_FIELDS = 40
CODE_ALIASES = {
    'ARTI': 'ART-I',
    'ART-I': 'ART-I',
    'ART': 'ART-I',
    'ENZO': 'ENZO',
    'Enzo': 'ENZO',
    'AREPO': 'AREPO',
    'GADGET3': 'GADGET-3',
    'GADGET-3': 'GADGET-3',
    'GEAR': 'GEAR',
    'CHANGA': 'CHANGA',
    'G4Cal_Pablo': 'GADGET-4',
    'G4CAL_PABLO': 'GADGET-4',
    'G4Cal': 'GADGET-4',
}

FOLDER_ALIASES = {
    'ART-I': 'ARTI',
    'ENZO': 'Enzo',
    'AREPO': 'AREPO',
    'GADGET-3': 'GADGET3',
    'GADGET-4': 'G4Cal_Pablo',
    'GEAR': 'GEAR',
    'CHANGA': 'CHANGA',
    'G4Cal_Pablo': 'GADGET-4',
    'G4CAL_PABLO': 'GADGET-4',
    'G4Cal': 'GADGET-4',
}

def normalize_user_code(code):
    if code not in CODE_ALIASES:
        raise ValueError(f'Unknown code {code!r}. Use one of: {sorted(CODE_ALIASES)}')
    return CODE_ALIASES[code]

# This notebook should be stored in the z0 directory.
ROOT = Path('/home/zhaozhang/local/AGORA_work/AGORA_Data/z0')
PIPELINE = ROOT / 'AGORA_parallel_DM_EM_projection_pipeline.py'

if not PIPELINE.exists():
    raise FileNotFoundError(f'Missing pipeline script: {PIPELINE}')

sys.path.insert(0, str(ROOT))
import AGORA_parallel_DM_EM_projection_pipeline as agora
agora = importlib.reload(agora)


def candidate_snapshots_for_code(code, root=ROOT):
    normalized = normalize_user_code(code)
    folder = root / FOLDER_ALIASES[normalized]
    if not folder.exists():
        raise FileNotFoundError(f'Missing folder for {normalized}: {folder}')

    if normalized == 'ART-I':
        candidates = sorted(folder.glob('*.d'))
    elif normalized == 'ENZO':
        candidates = sorted(p for p in folder.glob('RD*/RD*') if p.is_file() and p.name == p.parent.name)
    elif normalized == 'AREPO':
        # Main AREPO snapshot is snap_###.hdf5. Exclude auxiliary files such as snap_###.hsml.hdf5.
        candidates = sorted(p for p in folder.glob('snap_*.hdf5') if '.hsml.' not in p.name and '.kdtree' not in p.name)
    elif normalized in {'GADGET-3', 'GADGET-4'}:
        # Multi-file Gadget snapshots should be opened from the .0.hdf5 member.
        candidates = (sorted(folder.glob('snapshot_*/*.0.hdf5')) + sorted(folder.glob('snapshot_*.hdf5')) + sorted(folder.glob('snap_*.hdf5')))
        candidates = [p for p in candidates if 'fof_subhalo' not in p.name]
    elif normalized == 'GEAR':
        candidates = sorted(p for p in (list(folder.glob('snapshot_*.hdf5')) + list(folder.glob('*.hdf5'))) if '.hsml.' not in p.name)
    elif normalized == 'CHANGA':
        # CHANGA/Tipsy main file is normally like ncal-IV.003524. Sidecars append names such as .HII or .massform.
        def is_changa_main_file(path):
            if not path.is_file() or not path.name.startswith('ncal-'):
                return False
            parts = path.name.split('.')
            return len(parts) == 2 and parts[-1].isdigit()
        preferred = sorted(p for p in folder.iterdir() if is_changa_main_file(p))
        fallback = sorted(
            p for p in folder.iterdir()
            if p.is_file() and not p.name.startswith('.') and p.name not in {'wget-log', 'robots.txt.tmp'}
            and not any(p.name.endswith(s) for s in ['.HII', '.massform', '.Metalsdot', '.ESNRate', '.kdtree'])
        )
        candidates = preferred or fallback
    else:
        candidates = []
    return normalized, folder, candidates

def discover_dataset(code, root=ROOT, index=0):
    normalized, folder, candidates = candidate_snapshots_for_code(code, root)
    if not candidates:
        raise FileNotFoundError(f'No candidate snapshot found for {normalized} in {folder}')
    snapshot = candidates[index]
    return {
        'input_code': code,
        'code': normalized,
        'folder': folder,
        'snapshot': snapshot,
        'candidate_count': len(candidates),
        'all_candidates': candidates,
    }

def _field_exists_for_inspection(ds, field):
    try:
        return field in ds.field_list or field in ds.derived_field_list
    except Exception:
        return False


def inspect_one_agora_dataset(dataset_code):
    selected = discover_dataset(dataset_code)
    code = selected["code"]
    snapshot = selected["snapshot"]
    norm_code = agora.normalize_code(code)

    print("\n" + "=" * 90)
    print(f"DATASET_CODE = {dataset_code}")
    print(f"normalized   = {norm_code}")
    print(f"snapshot     = {snapshot}")
    print(f"candidates   = {selected['candidate_count']}")

    ds = agora.load_dataset(str(snapshot), norm_code, "auto")
    code_cfg = agora.AGORA_CODE_CONFIG.get(norm_code, agora.default_config_for_unknown(norm_code, ds))

    print("\n[Units / cosmology]")
    for k, v in agora.dataset_unit_metadata(ds).items():
        print(f"  {k}: {v}")

    print("\n[yt dataset]")
    print("  dataset_type:", getattr(ds, "dataset_type", None))
    print("  particle_types:", getattr(ds, "particle_types", None))
    print("  particle_types_raw:", getattr(ds, "particle_types_raw", None))
    print("  domain_dimensions:", getattr(ds, "domain_dimensions", None))

    try:
        print("  field_count:", len(ds.field_list))
        print("  derived_field_count:", len(ds.derived_field_list))
    except Exception as exc:
        print("  field list load failed:", repr(exc))

    try:
        print("  particle_type_counts:", getattr(ds, "particle_type_counts", None))
    except Exception:
        pass

    print("\n[Chosen field types]")
    gas_ftype = agora.choose_ftype(ds, "auto", code_cfg.get("gas_types", ["gas"]), code_cfg.get("density_names", ["density"]))
    star_ftype = agora.choose_ftype(ds, "auto", code_cfg.get("star_types", []), ["particle_position_x", "particle_position", "Coordinates", "x"])
    print("  gas_ftype :", gas_ftype)
    print("  star_ftype:", star_ftype)
    if norm_code == "ENZO" and star_ftype == "all":
        print("  NOTE: Enzo 'all' is an aggregate particle container, not a clean stellar disk tracer.")

    print("\n[Key gas fields]")
    key_groups = {
        "density": code_cfg.get("density_names", []),
        "temperature": code_cfg.get("temperature_names", []),
        "electron": code_cfg.get("electron_names", []),
        "HII": code_cfg.get("hii_names", []),
        "HeII": code_cfg.get("heii_names", []),
        "HeIII": code_cfg.get("heiii_names", []),
        "mass": code_cfg.get("gas_mass_names", []),
        "smoothing_length": code_cfg.get("smoothing_length_names", []),
    }
    for label, names in key_groups.items():
        found = agora.first_existing_field(ds, code_cfg.get("gas_types", ["gas"]), names) if names else None
        print(f"  {label:16s}: {found}")

    print("\n[Key star fields]")
    for label, names in {
        "mass": code_cfg.get("star_mass_names", []),
        "position_x": ["particle_position_x", "x"],
        "position_vec": ["particle_position", "Coordinates"],
        "velocity_vec": code_cfg.get("velocity_vector_names", ["particle_velocity", "Velocities"]),
    }.items():
        found = agora.first_existing_field(ds, code_cfg.get("star_types", []), names) if names else None
        print(f"  {label:16s}: {found}")

    print("\n[Sample raw field names]")
    try:
        shown = 0
        for f in ds.field_list:
            if shown >= INSPECT_MAX_FIELDS:
                break
            print(" ", f)
            shown += 1
    except Exception as exc:
        print("  could not print field_list:", repr(exc))

    return ds


inspection_results = {}
for dataset_code in INSPECT_CODES:
    try:
        inspection_results[dataset_code] = inspect_one_agora_dataset(dataset_code)
    except Exception as exc:
        print("\n" + "=" * 90)
        print(f"FAILED inspecting {dataset_code}: {exc!r}")
